In [1]:
# 2026-05-29 08:30 Europe/Rome — PROMPT: "riscrivi notebook: leggere file Excel con ticker, primo ticker benchmark, download da yfinance"
# =============================
# CELL 1 - INSTALL, IMPORT, DRIVE
# =============================

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Google Drive mount skipped or unavailable:', e)

import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Installa solo se necessario
required_packages = {
    'yfinance': 'yfinance',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
    'reportlab': 'reportlab',
    'pypdf': 'pypdf'
}

for import_name, pip_name in required_packages.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

print('Setup completed.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup completed.


In [2]:
# 2026-05-29 08:30 Europe/Rome — PROMPT: "riscrivi notebook: leggere Tickers.xlsx con ticker, primo ticker benchmark, download da yfinance"
# =============================
# CELL 2 - PARAMETRI GENERALI
# =============================

from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# =============================
# CARTELLE PROGETTO
# =============================

def project_base_dir():
    drive_base = Path("/content/drive/MyDrive/TradingAlgoMosaic")
    local_base = Path.cwd()
    if (local_base / "Tickers.xlsx").exists():
        return local_base
    return drive_base if drive_base.exists() else local_base

BASE_DIR = project_base_dir()
BASE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RUN_MODE: "full" esegue i batch; "preview" mostra solo una pagina grafica senza download dati.
RUN_MODE = "full"

# Mercati da elaborare.
# Lascia vuoto per scegliere con prompt testuale dopo la lettura di Tickers.xlsx.
# Esempi: "Italy", "S&P100,Italy", "1,3", oppure "all".
MARKETS_TO_PROCESS = ""
PROCESS_ALL_MARKETS = False

# Aggiornamento sito: copia i report PDF prodotti dal batch nel progetto TradingAlgo.it.
# In Colab i due progetti devono stare nella stessa cartella Drive:
# /content/drive/MyDrive/TradingAlgoMosaic e /content/drive/MyDrive/TradingAlgo.it
UPDATE_SITE_REPORTS = True
SITE_PROJECT_DIR = BASE_DIR.parent / "TradingAlgo.it"
SITE_REPORTS_DIR = SITE_PROJECT_DIR / "public" / "reports"
# Link stabili usati dal sito. Il batch aggiorna il file locale con lo stesso nome;
# Google Drive mantiene lo stesso link quando il file viene sovrascritto dalla cartella sincronizzata.
SITE_REPORT_URLS = {
    "USA": "https://drive.google.com/file/d/1krZJUWeQlGR1KTKKg1g0qBDA7mtKmNEQ/view?usp=drive_link",
    "Europe": "https://drive.google.com/file/d/1PyQVkBjpdQJ-u3A8CTPtZTxUgw7G07N0/view?usp=drive_link",
    "Italy": "https://drive.google.com/file/d/1vuY1MQdQc6OKyEcm1I7Vfm60XvZobnRi/view?usp=drive_link",
    "S&P100": "https://drive.google.com/file/d/1zOSpgM2qaB4oohb7RZbkr1TmXPzv-p91/view?usp=drive_link",
    "EuroStoxx50": "https://drive.google.com/file/d/1sPMuM8pfRPhZXlPLz05cE6X6lZtuJ6ym/view?usp=drive_link",
}


# =============================
# FILE TICKER
# =============================
# In Colab metti questo notebook e Tickers.xlsx nella stessa cartella:
# /content/drive/MyDrive/TradingAlgoMosaic
# Formato Excel: ogni colonna = mercato; riga 1 Market, riga 2 Benchmark, righe successive Tickers.
TICKERS_XLSX_PATH = BASE_DIR / "Tickers.xlsx"
if not TICKERS_XLSX_PATH.exists():
    raise FileNotFoundError(f"Tickers.xlsx non trovato in BASE_DIR: {BASE_DIR}")
DYNAMIC_ALLOCATION_PDF_PATH = OUTPUT_DIR / "DynamicRiskAllocation.pdf"
CREATE_DYNAMIC_RISK_ALLOCATION_TABLE = False
PRODUCTION_MARKET_REQUEST = ["USA", "Europe", "Italy"]
PRODUCTION_MARKET_ALIASES = {
    "USA": ["USA", "SP100", "S&P100", "S AND P100"],
    "Europe": ["Europe", "EXU", "EUROSTOXX50", "EURO STOXX 50", "EUROSTOXX", "EUROSTOXX50"],
    "Italy": ["Italy", "EWI"],
}

# =============================
# DOWNLOAD DATI
# =============================

START_DATE = "2023-01-01"
PERFORMANCE_START_DATE = "2024-01-01"
def last_available_friday(today=None):
    today = today or datetime.now(ZoneInfo("Europe/Rome")).date()
    days_since_friday = (today.weekday() - 4) % 7
    return today - timedelta(days=days_since_friday)

LAST_AVAILABLE_FRIDAY = last_available_friday()
# yfinance usa end come data esclusiva: +1 giorno per includere l'ultimo venerdi disponibile.
END_DATE = (LAST_AVAILABLE_FRIDAY + timedelta(days=1)).isoformat()

PRICE_FIELD_PRIORITY = [
    "Adj Close",
    "Close"
]

# =============================
# FREQUENZA
# =============================

WEEKLY_RULE = "W-FRI"

# =============================
# MODELLO
# =============================

K_SELECT_LIST = [5]

RISK_FREE_ANNUAL = 0.01

TRANSACTION_COST = 0.001

# =============================
# HEDGING DI PORTAFOGLIO
# =============================
# Hedge settimanale sul benchmark del mercato: la size short cresce quando
# Trend_Score + Stability_Score dei titoli selezionati si indeboliscono.
ENABLE_BENCHMARK_HEDGE = True
HEDGE_MAX_SHORT = 1.00
HEDGE_MIN_SCORE = 0.0
HEDGE_MAX_SCORE = 100.0
HEDGE_SCORE_TREND_WEIGHT = 0.55
HEDGE_SCORE_STABILITY_WEIGHT = 0.45

# =============================
# GRID SEARCH
# =============================

TREND_LEN_LIST = [52]

SLOPE_MIN_LIST = [
    0.00,

]

R2_MIN_LIST = [

    0.30
]

CORR_MAX_LIST = [
    0.5,

]

CORR_LOOKBACK = 26

# =============================
# CHECK
# =============================

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TICKERS_XLSX_PATH:", TICKERS_XLSX_PATH)
print("RUN_MODE:", RUN_MODE)


BASE_DIR: /content/drive/MyDrive/TradingAlgoMosaic
OUTPUT_DIR: /content/drive/MyDrive/TradingAlgoMosaic/output
TICKERS_XLSX_PATH: /content/drive/MyDrive/TradingAlgoMosaic/Tickers.xlsx
RUN_MODE: full


In [3]:
# 2026-06-03 Europe/Rome - batch multi-mercato: selezione mercati stabile per Colab
# =============================
# CELL 3 - LETTURA EXCEL + INPUT MARKET
# =============================

def clean_ticker_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    txt = str(value).strip().upper()
    if txt == "" or txt.lower() == "nan":
        return None
    return txt


def read_markets_from_excel(xlsx_path):
    # Formato Tickers.xlsx: riga 1 = Market, riga 2 = Benchmark, righe successive = Tickers.
    xlsx_path = Path(xlsx_path)
    if not xlsx_path.exists():
        raise FileNotFoundError(f"File Excel ticker non trovato: {xlsx_path}")

    raw = pd.read_excel(xlsx_path, header=None, engine="openpyxl")
    if raw.empty or raw.shape[0] < 3:
        raise ValueError("Tickers.xlsx deve avere almeno 3 righe: Market, Benchmark, Tickers.")

    markets = []
    for col in raw.columns:
        market = raw.iloc[0, col]
        benchmark_value = raw.iloc[1, col]
        if pd.isna(market) or str(market).strip() == "":
            continue

        market_name = str(market).strip()
        benchmark = clean_ticker_value(benchmark_value)
        tickers = []
        for value in raw.iloc[2:, col].tolist():
            ticker = clean_ticker_value(value)
            if ticker is not None:
                tickers.append(ticker)
        tickers = list(dict.fromkeys(tickers))

        if benchmark is None:
            print(f"Mercato ignorato senza benchmark: {market_name}")
            continue
        if len(tickers) == 0:
            print(f"Mercato ignorato senza ticker investibili: {market_name}")
            continue

        markets.append({
            "Market": market_name,
            "Benchmark": benchmark,
            "Universe": tickers,
            "Ticker_Count": len(tickers),
            "Column": col,
        })

    if not markets:
        raise ValueError("Nessun mercato valido trovato in Tickers.xlsx.")
    return markets, raw


def _normalize_market_key(value):
    txt = str(value).upper().strip()
    return "".join(ch for ch in txt if ch.isalnum())


def print_available_markets(markets):
    print("\nMERCATI DISPONIBILI IN Tickers.xlsx")
    print("-" * 80)
    for i, item in enumerate(markets, start=1):
        print(f"{i}. {item['Market']} | Benchmark: {item['Benchmark']} | Ticker: {item['Ticker_Count']}")
    print("-" * 80)


def select_markets_from_value(markets, value):
    value = str(value or "").strip()
    if value.lower() in {"all", "tutti", "tutto", "*"}:
        return [item.copy() for item in markets]

    by_name = {_normalize_market_key(item["Market"]): item for item in markets}
    selected = []
    seen = set()

    for token in value.replace(";", ",").split(","):
        token = token.strip()
        if not token:
            continue

        item = None
        if token.isdigit():
            idx = int(token)
            if 1 <= idx <= len(markets):
                item = markets[idx - 1]
        else:
            item = by_name.get(_normalize_market_key(token))

        if item is None:
            available = ", ".join(item["Market"] for item in markets)
            raise ValueError(f"Mercato '{token}' non trovato. Disponibili: {available}")

        key = item["Market"]
        if key not in seen:
            selected.append(item.copy())
            seen.add(key)

    return selected


def choose_markets_for_batch(markets):
    print_available_markets(markets)

    if bool(globals().get("PROCESS_ALL_MARKETS", False)):
        return [item.copy() for item in markets]

    configured_value = str(globals().get("MARKETS_TO_PROCESS", "") or "").strip()
    if configured_value:
        return select_markets_from_value(markets, configured_value)

    choice = input("Mercati da elaborare (es. 1,3 oppure Italy,S&P100 oppure all): ").strip()
    selected = select_markets_from_value(markets, choice)
    if not selected:
        raise ValueError("Nessun mercato selezionato.")
    return selected


available_markets, sheet_raw = read_markets_from_excel(TICKERS_XLSX_PATH)
selected_market_infos = choose_markets_for_batch(available_markets)

print("\nMERCATI SELEZIONATI PER IL BATCH:")
for item in selected_market_infos:
    print(f"- {item['Market']} | Benchmark: {item['Benchmark']} | Ticker: {item['Ticker_Count']}")

try:
    display(sheet_raw.head(10))
except Exception:
    print(sheet_raw.head(10))



MERCATI DISPONIBILI IN Tickers.xlsx
--------------------------------------------------------------------------------
1. USA100 | Benchmark: SPY | Ticker: 100
2. Europe50 | Benchmark: ^STOXX50E | Ticker: 49
3. Italy30 | Benchmark: FTSEMIB.MI | Ticker: 28
4. India30 | Benchmark: ^NSEI | Ticker: 31
5. UK30 | Benchmark: ^FTSE | Ticker: 30
6. France40 | Benchmark: ^FCHI | Ticker: 40
7. Germany30 | Benchmark: ^GDAXI | Ticker: 40
8. Israel30 | Benchmark: TA35.TA | Ticker: 35
9. Emirates50 | Benchmark: DFMGI.AE | Ticker: 45
10. Australia50 | Benchmark: ^AXJO | Ticker: 50
11. Japan50 | Benchmark: ^N225 | Ticker: 50
12. China50 | Benchmark: 000300.SS | Ticker: 50
13. Canada50 | Benchmark: ^GSPTSE | Ticker: 50
14. Argentina30 | Benchmark: ^MERV | Ticker: 30
15. Mexico30 | Benchmark: ^MXX | Ticker: 35
16. Brazil50 | Benchmark: ^BVSP | Ticker: 50
17. South Korea30 | Benchmark: ^KS11 | Ticker: 29
18. South Africa30 | Benchmark: ^J203.JO | Ticker: 26
-------------------------------------------------

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,USA100,Europe50,Italy30,India30,UK30,France40,Germany30,Israel30,Emirates50,Australia50,Japan50,China50,Canada50,Argentina30,Mexico30,Brazil50,South Korea30,South Africa30
1,SPY,^STOXX50E,FTSEMIB.MI,^NSEI,^FTSE,^FCHI,^GDAXI,TA35.TA,DFMGI.AE,^AXJO,^N225,000300.SS,^GSPTSE,^MERV,^MXX,^BVSP,^KS11,^J203.JO
2,AAPL,ADYEN.AS,UCG.MI,HDFCBANK.NS,AZN.L,AC.PA,SIE.DE,BEZQ.TA,EMAAR.AE,ALL.AX,7203.T,601398.SS,RY.TO,YPFD.BA,GMEXICOB.MX,PETR4.SA,005930.KS,NPN.JO
3,ABBV,AD.AS,ISP.MI,ICICIBANK.NS,SHEL.L,AI.PA,DTE.DE,CLIS.TA,FAB.AD,AMC.AX,8306.T,601857.SS,TD.TO,GGAL.BA,AMXB.MX,VALE3.SA,000660.KS,PRX.JO
4,ABT,AI.PA,BAMI.MI,SBIN.NS,HSBA.L,AIR.PA,ALV.DE,NWMD.TA,EAND.AD,ANZ.AX,6758.T,600519.SS,SHOP.TO,PAMP.BA,WALMEX.MX,ITUB4.SA,373220.KS,CFR.JO
5,ACN,AIR.PA,MB.MI,KOTAKBANK.NS,ULVR.L,ALO.PA,MUV2.DE,DLEKG.TA,EMIRATESNBD.AE,ASX.AX,6861.T,601988.SS,ENB.TO,TXAR.BA,FEMSAUBD.MX,BBDC4.SA,207940.KS,BTI.JO
6,ADBE,ALV.DE,BMED.MI,AXISBANK.NS,BP.L,ATO.PA,DHL.DE,DSCT.TA,ADCB.AD,BHP.AX,9984.T,601088.SS,BMO.TO,TGSU2.BA,GFNORTEO.MX,ABEV3.SA,005380.KS,AGL.JO
7,ALL,ASML.AS,G.MI,BAJFINANCE.NS,GSK.L,BN.PA,BMW.DE,ESLT.TA,ADIB.AD,BXB.AX,8035.T,600036.SS,BN.TO,ALUA.BA,CEMEXCPO.MX,BBAS3.SA,000270.KS,ANH.JO
8,AMAT,ABI.BR,UNI.MI,TCS.NS,RIO.L,BNP.PA,BAS.DE,FIBI.TA,ADNOCDRILL.AD,CAR.AX,6098.T,601628.SS,CNQ.TO,COME.BA,BIMBOA.MX,WEGE3.SA,035420.KS,FSR.JO
9,AMD,BAS.DE,ENI.MI,INFY.NS,REL.L,CA.PA,DB1.DE,HARL.TA,DIB.AE,CBA.AX,9432.T,601318.SS,BNS.TO,BMA.BA,TLEVISACPO.MX,B3SA3.SA,068270.KS,SBK.JO


In [4]:
# 2026-05-29 08:30 Europe/Rome — PROMPT: "riscrivi notebook: leggere file Excel con ticker, primo ticker benchmark, download da yfinance"
# =============================
# CELL 4 - DOWNLOAD DATI DA YFINANCE
# =============================

def extract_price_table(raw, tickers, field_priority=None):
    # Estrae una tabella prezzi robusta da yfinance, gestendo colonne MultiIndex e campi mancanti.
    if field_priority is None:
        field_priority = ['Adj Close', 'Close']

    if raw is None or raw.empty:
        raise ValueError('Download yfinance vuoto.')

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = list(raw.columns.get_level_values(0).unique())
        level1 = list(raw.columns.get_level_values(1).unique())

        # Caso standard yfinance group_by='column': livello 0 = campi, livello 1 = ticker
        for field in field_priority:
            if field in level0:
                px = raw[field].copy()
                px.columns = [str(c).upper() for c in px.columns]
                return px

        # Caso alternativo: livello 0 = ticker, livello 1 = campi
        for field in field_priority:
            if field in level1:
                px = raw.xs(field, level=1, axis=1).copy()
                px.columns = [str(c).upper() for c in px.columns]
                return px

        raise ValueError(f'Nessun campo prezzo trovato tra: {field_priority}')

    # Caso singolo ticker o colonne semplici
    for field in field_priority:
        if field in raw.columns:
            first_ticker = tickers[0]
            return pd.DataFrame({first_ticker: raw[field]})

    raise ValueError(f'Nessun campo prezzo trovato tra: {field_priority}')



def download_prices_yfinance(tickers, start_date, end_date=None, benchmark_ticker=None, market_name=None):
    # In produzione Colab scarica sempre da yfinance fino all'ultimo venerdi disponibile.
    print(f"Download yfinance: {market_name or 'market'} | start={start_date} | end_exclusive={end_date}")
    raw = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
        group_by='column',
        threads=True
    )

    benchmark_check = benchmark_ticker if benchmark_ticker is not None else tickers[0]

    px = extract_price_table(raw, tickers, PRICE_FIELD_PRIORITY)
    px = px.sort_index()
    px.index = pd.to_datetime(px.index).tz_localize(None)

    # Mantiene solo i ticker richiesti e nell'ordine corretto, se presenti
    available = [t for t in tickers if t in px.columns]
    missing = [t for t in tickers if t not in px.columns]
    px = px[available]

    # Rimuove colonne completamente vuote
    px = px.dropna(axis=1, how='all')
    missing_after = [t for t in tickers if t not in px.columns]

    if missing_after:
        print('Ticker senza dati o non scaricati:', missing_after)

    if benchmark_check not in px.columns:
        raise ValueError(f'Benchmark {benchmark_check} non presente nei dati scaricati.')

    return px


In [5]:
# 2026-06-03 20:34 Europe/Rome — PROMPT: "controlla elaborazione batch: serve solo pdf"
# =============================
# CELL 5 - PREPARAZIONE DATI SETTIMANALI
# =============================

def prepare_weekly_prices(prices_daily, benchmark, universe, weekly_rule='W-FRI'):
    # Converte i prezzi giornalieri in prezzi settimanali e allinea benchmark/universo.
    prices_weekly = prices_daily.resample(weekly_rule).last().dropna(how='all')
    prices_weekly = prices_weekly.ffill()

    available_universe = [t for t in universe if t in prices_weekly.columns]
    if not available_universe:
        raise ValueError('Nessun ticker investibile disponibile dopo il download.')

    if benchmark not in prices_weekly.columns:
        raise ValueError(f'Benchmark {benchmark} non presente nei prezzi settimanali.')

    px_bench = prices_weekly[benchmark].dropna()
    px_univ = prices_weekly[available_universe].dropna(how='all').ffill()

    common_index = px_bench.index.intersection(px_univ.index)
    px_bench = px_bench.loc[common_index]
    px_univ = px_univ.loc[common_index]

    raw_returns_univ = px_univ.pct_change()
    max_abs_weekly_return = raw_returns_univ.abs().max(skipna=True)
    bad_tickers = max_abs_weekly_return[max_abs_weekly_return > 3.0].index.tolist()
    if bad_tickers:
        print("Ticker rimossi per spike settimanali anomali > 300%:", bad_tickers)
        px_univ = px_univ.drop(columns=bad_tickers)
        if px_univ.empty:
            raise ValueError("Nessun ticker investibile dopo il filtro anti-spike.")

    returns_univ = px_univ.pct_change().fillna(0.0)
    returns_bench = px_bench.pct_change().fillna(0.0)

    return px_univ, px_bench, returns_univ, returns_bench


In [6]:
# 2026-05-29 08:30 Europe/Rome — PROMPT: "riscrivi notebook: leggere file Excel con ticker, primo ticker benchmark, download da yfinance"
# =============================
# CELL 6 - FUNZIONI TREND, R2, METRICHE
# =============================

def regression_slope_r2(values):
    # Calcola slope e R2 su log-prezzi usando regressione lineare semplice.
    y = np.asarray(values, dtype=float)
    if len(y) < 3 or np.any(~np.isfinite(y)) or np.any(y <= 0):
        return np.nan, np.nan

    y = np.log(y)
    x = np.arange(len(y), dtype=float)
    x_mean = x.mean()
    y_mean = y.mean()

    den = np.sum((x - x_mean) ** 2)
    if den == 0:
        return np.nan, np.nan

    slope = np.sum((x - x_mean) * (y - y_mean)) / den
    intercept = y_mean - slope * x_mean
    y_hat = intercept + slope * x

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y_mean) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return slope, r2


def max_drawdown(equity):
    # Calcola il massimo drawdown di una equity line.
    equity = pd.Series(equity).dropna()
    dd = equity / equity.cummax() - 1.0
    return dd.min()


def performance_metrics(strategy_ret, benchmark_ret, risk_free_annual=0.01):
    # Calcola metriche standard settimanali della strategia.
    strategy_ret = pd.Series(strategy_ret).dropna()
    benchmark_ret = pd.Series(benchmark_ret).reindex(strategy_ret.index).fillna(0.0)

    periods = 52
    rf_weekly = (1 + risk_free_annual) ** (1 / periods) - 1

    equity = (1 + strategy_ret).cumprod()
    bench_equity = (1 + benchmark_ret).cumprod()

    years = len(strategy_ret) / periods
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = strategy_ret.std() * np.sqrt(periods)
    sharpe = ((strategy_ret - rf_weekly).mean() / strategy_ret.std()) * np.sqrt(periods) if strategy_ret.std() > 0 else np.nan

    active_ret = strategy_ret - benchmark_ret
    ir = (active_ret.mean() / active_ret.std()) * np.sqrt(periods) if active_ret.std() > 0 else np.nan

    return {
        'CAGR': cagr,
        'Volatility': vol,
        'Sharpe_1pct': sharpe,
        'Information_Ratio': ir,
        'MaxDD': max_drawdown(equity),
        'Final_Equity': equity.iloc[-1],
        'Benchmark_Final_Equity': bench_equity.iloc[-1]
    }

In [7]:
# 2026-05-29 08:30 Europe/Rome — PROMPT: "riscrivi notebook: leggere file Excel con ticker, primo ticker benchmark, download da yfinance"
# =============================
# CELL 7 - COSTRUZIONE PORTAFOGLIO SETTIMANALE
# =============================

def select_portfolio_at_date(px_univ, date, trend_len, slope_min, r2_min, corr_max, corr_lookback, k_select):
    # Seleziona ticker con trend positivo/stabile e filtro di correlazione.
    loc = px_univ.index.get_loc(date)
    if loc < max(trend_len, corr_lookback):
        return [], pd.DataFrame()

    hist_trend = px_univ.iloc[loc - trend_len + 1:loc + 1]
    hist_corr = px_univ.iloc[loc - corr_lookback + 1:loc + 1].pct_change().dropna(how='all')

    rows = []
    for ticker in px_univ.columns:
        s = hist_trend[ticker].dropna()
        if len(s) < trend_len:
            continue
        slope, r2 = regression_slope_r2(s.values)
        rows.append({
            'Ticker': ticker,
            'Slope': slope,
            'R2': r2,
            'Score': slope * max(r2, 0) if pd.notna(slope) and pd.notna(r2) else np.nan
        })

    ranking = pd.DataFrame(rows)
    if ranking.empty:
        return [], ranking

    valid_slope = ranking['Slope'].replace([np.inf, -np.inf], np.nan)
    if valid_slope.notna().any():
        slope_rank = valid_slope.rank(pct=True).fillna(0.0) * 100.0
    else:
        slope_rank = pd.Series(0.0, index=ranking.index)
    ranking['Trend_Score'] = slope_rank.clip(0, 100)
    ranking['Stability_Score'] = (ranking['R2'].fillna(0.0).clip(lower=0.0, upper=1.0) * 100.0)
    ranking['Portfolio_Contribution_Score'] = (0.55 * ranking['Trend_Score'] + 0.45 * ranking['Stability_Score']).clip(0, 100)
    ranking['Pass_Trend'] = ranking['Slope'] >= slope_min
    ranking['Pass_Stability'] = ranking['R2'] >= r2_min
    ranking['Candidate'] = ranking['Pass_Trend'] & ranking['Pass_Stability']
    ranking = ranking.sort_values(['Candidate', 'Score', 'Slope', 'R2'], ascending=[False, False, False, False])

    selected = []
    corr = hist_corr.corr()

    for _, row in ranking[ranking['Candidate']].iterrows():
        ticker = row['Ticker']
        if len(selected) == 0:
            selected.append(ticker)
        else:
            max_corr = corr.loc[ticker, selected].max() if ticker in corr.index else np.nan
            if pd.isna(max_corr) or max_corr <= corr_max:
                selected.append(ticker)

        if len(selected) >= k_select:
            break

    ranking['Selected'] = ranking['Ticker'].isin(selected)
    return selected, ranking


def run_strategy(px_univ, ret_univ, ret_bench, trend_len, slope_min, r2_min, corr_max, corr_lookback, k_select, transaction_cost):
    # Esegue backtest settimanale con pesi equal-weight e cash se i selezionati sono meno di K.
    weights = pd.DataFrame(0.0, index=px_univ.index, columns=list(px_univ.columns) + ['CASH'])
    hedge_targets = pd.Series(0.0, index=px_univ.index, name='Hedge_Short_Target')
    hedge_scores = pd.Series(np.nan, index=px_univ.index, name='Hedge_Portfolio_Score')
    selected_by_date = {}
    ranking_by_date = {}

    for date in px_univ.index:
        selected, ranking = select_portfolio_at_date(
            px_univ, date, trend_len, slope_min, r2_min, corr_max, corr_lookback, k_select
        )
        selected_by_date[date] = selected
        ranking_by_date[date] = ranking

        if selected:
            w = 1.0 / k_select
            for ticker in selected:
                weights.loc[date, ticker] = w
            weights.loc[date, 'CASH'] = max(0.0, 1.0 - len(selected) / k_select)
        else:
            weights.loc[date, 'CASH'] = 1.0

        if bool(globals().get('ENABLE_BENCHMARK_HEDGE', True)) and selected and not ranking.empty:
            selected_ranking = ranking[ranking['Ticker'].isin(selected)].copy()
            trend_weight = float(globals().get('HEDGE_SCORE_TREND_WEIGHT', 0.55))
            stability_weight = float(globals().get('HEDGE_SCORE_STABILITY_WEIGHT', 0.45))
            denom = trend_weight + stability_weight
            if denom <= 0:
                trend_weight, stability_weight, denom = 0.55, 0.45, 1.0
            if 'Portfolio_Contribution_Score' not in selected_ranking.columns:
                trend_score = selected_ranking['Trend_Score'] if 'Trend_Score' in selected_ranking.columns else pd.Series(0.0, index=selected_ranking.index)
                stability_score = selected_ranking['Stability_Score'] if 'Stability_Score' in selected_ranking.columns else pd.Series(0.0, index=selected_ranking.index)
                selected_ranking['Portfolio_Contribution_Score'] = (
                    trend_weight * trend_score.fillna(0.0)
                    + stability_weight * stability_score.fillna(0.0)
                ) / denom
            portfolio_score = selected_ranking['Portfolio_Contribution_Score'].astype(float).mean()
            min_score = float(globals().get('HEDGE_MIN_SCORE', 0.0))
            max_score = float(globals().get('HEDGE_MAX_SCORE', 100.0))
            if max_score <= min_score:
                min_score, max_score = 0.0, 100.0
            normalized_score = (portfolio_score - min_score) / (max_score - min_score)
            hedge_intensity = 1.0 - float(np.clip(normalized_score, 0.0, 1.0))
            long_exposure = float(weights.loc[date, px_univ.columns].sum())
            hedge_targets.loc[date] = min(float(globals().get('HEDGE_MAX_SHORT', 1.0)), max(0.0, long_exposure * hedge_intensity))
            hedge_scores.loc[date] = portfolio_score

    asset_cols = px_univ.columns.tolist()
    prev_weights = weights[asset_cols].shift(1).fillna(0.0)
    prev_hedge = hedge_targets.shift(1).fillna(0.0)

    asset_return_contributions = prev_weights * ret_univ[asset_cols]
    gross_ret = asset_return_contributions.sum(axis=1)
    turnover = weights[asset_cols].diff().abs().sum(axis=1).fillna(0.0)
    costs = turnover * transaction_cost
    strategy_ret = gross_ret - costs
    hedge_turnover = hedge_targets.diff().abs().fillna(0.0)
    hedge_costs = hedge_turnover * transaction_cost
    benchmark_ret = ret_bench.reindex(strategy_ret.index).fillna(0.0)
    hedge_ret = -prev_hedge * benchmark_ret
    hedged_strategy_ret = strategy_ret + hedge_ret - hedge_costs

    result = pd.DataFrame({
        'Strategy_Return': strategy_ret,
        'Hedged_Strategy_Return': hedged_strategy_ret,
        'Benchmark_Return': benchmark_ret,
        'Gross_Return': gross_ret,
        'Hedge_Return': hedge_ret,
        'Transaction_Cost': costs,
        'Hedge_Cost': hedge_costs,
        'Turnover': turnover,
        'Hedge_Turnover': hedge_turnover,
        'Exposure': weights[asset_cols].sum(axis=1),
        'Net_Exposure_After_Hedge': weights[asset_cols].sum(axis=1) - hedge_targets,
        'Hedge_Short_Target': hedge_targets,
        'Hedge_Portfolio_Score': hedge_scores,
        'CASH': weights['CASH']
    })

    result['Strategy_Equity'] = (1 + result['Strategy_Return']).cumprod()
    result['Hedged_Strategy_Equity'] = (1 + result['Hedged_Strategy_Return']).cumprod()
    result['Benchmark_Equity'] = (1 + result['Benchmark_Return']).cumprod()

    equity_before_period = result['Strategy_Equity'].shift(1).fillna(1.0)
    asset_profit_contributions = asset_return_contributions.mul(equity_before_period, axis=0)
    ticker_profit_contributors = (
        asset_profit_contributions.sum(axis=0)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values(ascending=False)
    )
    result['Strategy_Drawdown'] = result['Strategy_Equity'] / result['Strategy_Equity'].cummax() - 1.0
    result['Hedged_Strategy_Drawdown'] = result['Hedged_Strategy_Equity'] / result['Hedged_Strategy_Equity'].cummax() - 1.0
    result['Benchmark_Drawdown'] = result['Benchmark_Equity'] / result['Benchmark_Equity'].cummax() - 1.0

    metrics = performance_metrics(result['Strategy_Return'], result['Benchmark_Return'], RISK_FREE_ANNUAL)
    hedged_metrics = performance_metrics(result['Hedged_Strategy_Return'], result['Benchmark_Return'], RISK_FREE_ANNUAL)
    metrics['Avg_Turnover'] = result['Turnover'].mean()
    metrics['Avg_Exposure'] = result['Exposure'].mean()
    metrics['Avg_Hedge_Short'] = result['Hedge_Short_Target'].mean()
    metrics['Last_Hedge_Short'] = result['Hedge_Short_Target'].iloc[-1]
    metrics['Last_Hedge_Score'] = result['Hedge_Portfolio_Score'].iloc[-1]
    metrics['Hedged_CAGR'] = hedged_metrics.get('CAGR')
    metrics['Hedged_Volatility'] = hedged_metrics.get('Volatility')
    metrics['Hedged_Sharpe_1pct'] = hedged_metrics.get('Sharpe_1pct')
    metrics['Hedged_Information_Ratio'] = hedged_metrics.get('Information_Ratio')
    metrics['Hedged_MaxDD'] = hedged_metrics.get('MaxDD')
    metrics['Hedged_Final_Equity'] = hedged_metrics.get('Final_Equity')

    return {
        'result': result,
        'weights': weights,
        'hedge_targets': hedge_targets,
        'selected_by_date': selected_by_date,
        'ranking_by_date': ranking_by_date,
        'ticker_profit_contributors': ticker_profit_contributors,
        'asset_profit_contributions': asset_profit_contributions,
        'metrics': metrics,
        'params': {
            'trend_len': trend_len,
            'slope_min': slope_min,
            'r2_min': r2_min,
            'corr_max': corr_max,
            'corr_lookback': corr_lookback,
            'k_select': k_select
        }
    }


In [8]:
# 2026-06-15 Europe/Rome ? production Colab: single configured run, grid search removed
# =============================
# CELL 8 - SINGLE RUN FUNCTION
# =============================

def run_single_configuration_for_market(px_univ, ret_univ, ret_bench):
    # Esegue una sola configurazione production, senza grid search.
    params = {
        "trend_len": TREND_LEN_LIST[0],
        "slope_min": SLOPE_MIN_LIST[0],
        "r2_min": R2_MIN_LIST[0],
        "corr_max": CORR_MAX_LIST[0],
        "corr_lookback": CORR_LOOKBACK,
        "k_select": K_SELECT_LIST[0],
    }
    run = run_strategy(
        px_univ=px_univ,
        ret_univ=ret_univ,
        ret_bench=ret_bench,
        trend_len=params["trend_len"],
        slope_min=params["slope_min"],
        r2_min=params["r2_min"],
        corr_max=params["corr_max"],
        corr_lookback=params["corr_lookback"],
        k_select=params["k_select"],
        transaction_cost=TRANSACTION_COST,
    )
    runs_df = pd.DataFrame([{"Run": 1, **run["params"], **run["metrics"]}])
    print("Production parameters:", run["params"])
    display(runs_df)
    return runs_df, {1: run}, run


def run_grid_search_for_market(px_univ, ret_univ, ret_bench):
    # Backward-compatible wrapper: grid search disattivata.
    return run_single_configuration_for_market(px_univ, ret_univ, ret_bench)


In [9]:
# 2026-06-17 Europe/Rome - production: 3 batch indipendenti, un PDF accorpato per mercato
# =============================
# CELL 10 - 3 BATCH INDIPENDENTI + REPORT ACCORPATO PER MERCATO
# =============================



# =============================
# PDF / REPORT HELPERS
# =============================
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak


def safe_text(value, default="-"):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    txt = str(value).strip()
    return txt if txt else default


def safe_report_market_name(value):
    txt = safe_text(value, "Report")
    return "".join(ch if ch.isalnum() else "_" for ch in txt).strip("_") or "Report"


def pdf_text(value, default="-"):
    import html
    txt = html.escape(safe_text(value, default)).replace("\n", "<br/>")
    return txt.replace("&lt;br/&gt;", "<br/>").replace("&lt;br&gt;", "<br/>").replace("&lt;br /&gt;", "<br/>")


def format_position_name(value, max_len=20):
    txt = safe_text(value, "")
    txt = " ".join(txt.lower().split()).title()
    return txt[:max_len].rstrip()


def format_position_category(value, max_len=17):
    txt = safe_text(value, "")
    if not txt or txt == "-":
        return "-"
    txt = " ".join(txt.lower().split()).capitalize()
    return txt[:max_len].rstrip()


def format_current_positions_for_pdf(value):
    import re
    if isinstance(value, list):
        parts = value
    else:
        raw = safe_text(value, "")
        if not raw or raw == "-":
            return "-"
        parts = [part.strip() for part in re.split(r"<br\s*/?>|\n", raw) if part.strip()]
    if not parts:
        return "-"
    rows = []
    for part in parts:
        sector = "-"
        industry = "-"
        if isinstance(part, dict):
            ticker = part.get("Ticker", "")
            name = part.get("Name", "")
            sector = part.get("Sector", "-")
            industry = part.get("Industry", "-")
        elif " - " in part:
            ticker, name = part.split(" - ", 1)
        else:
            ticker, name = part, ""
        ticker_txt = pdf_text(str(ticker).upper())
        name_txt = pdf_text(format_position_name(name or ticker))
        category_txt = pdf_text(f"{format_position_category(sector, 13)} / {format_position_category(industry, 15)}")
        rows.append(f"<b>{ticker_txt}</b><br/>{name_txt}<br/><font size='5.8'>{category_txt}&nbsp;</font>")
    return "<br/><br/>".join(rows) if rows else "-"


def format_pct_for_pdf(value):
    try:
        v = float(value)
        if not np.isfinite(v):
            return "-"
        return f"{v * 100:.2f}%"
    except Exception:
        return "-"


def format_num_for_pdf(value):
    try:
        v = float(value)
        if not np.isfinite(v):
            return "-"
        return f"{v:.2f}"
    except Exception:
        return "-"


def normalize_dynamic_asset_label(asset):
    label = safe_text(asset)
    replacements = {
        "US100 - United States large-cap equity allocation": "US100",
        "Europe - Eurozone equity allocation": "Europe",
        "Italy - Italy equity allocation": "Italy",
        "GOLD - Gold allocation": "GOLD",
        "SPY": "US100",
        "EXU": "Europe",
        "EZU": "Europe",
        "EWI": "Italy",
    }
    return replacements.get(label, label)


def find_logo_file():
    for candidate in [BASE_DIR / "logo.png", BASE_DIR / "Production" / "logo.png"]:
        if candidate.exists():
            return candidate
    return None


def _safe_float(value, default=np.nan):
    try:
        v = float(value)
        return v if np.isfinite(v) else default
    except Exception:
        return default


def filter_result_for_performance_window(result_df, performance_start_date):
    if result_df is None or result_df.empty:
        return result_df
    start = pd.to_datetime(performance_start_date)
    filtered = result_df.loc[result_df.index >= start].copy()
    if filtered.empty:
        return filtered
    # Rebase all equity curves to the same first report point. The first
    # row is a baseline, so performance starts from the following period.
    if "Strategy_Return" in filtered.columns:
        strategy_returns = filtered["Strategy_Return"].fillna(0.0).copy()
        if not strategy_returns.empty:
            strategy_returns.iloc[0] = 0.0
        filtered["Strategy_Return"] = strategy_returns
        filtered["Strategy_Equity"] = (1 + strategy_returns).cumprod()
        filtered["Strategy_Drawdown"] = filtered["Strategy_Equity"] / filtered["Strategy_Equity"].cummax() - 1.0
    if "Hedged_Strategy_Return" in filtered.columns:
        hedged_returns = filtered["Hedged_Strategy_Return"].fillna(0.0).copy()
        if not hedged_returns.empty:
            hedged_returns.iloc[0] = 0.0
        filtered["Hedged_Strategy_Return"] = hedged_returns
        filtered["Hedged_Strategy_Equity"] = (1 + hedged_returns).cumprod()
        filtered["Hedged_Strategy_Drawdown"] = filtered["Hedged_Strategy_Equity"] / filtered["Hedged_Strategy_Equity"].cummax() - 1.0
    if "Benchmark_Return" in filtered.columns:
        benchmark_returns = filtered["Benchmark_Return"].fillna(0.0).copy()
        if not benchmark_returns.empty:
            benchmark_returns.iloc[0] = 0.0
        filtered["Benchmark_Return"] = benchmark_returns
        filtered["Benchmark_Equity"] = (1 + benchmark_returns).cumprod()
        filtered["Benchmark_Drawdown"] = filtered["Benchmark_Equity"] / filtered["Benchmark_Equity"].cummax() - 1.0
    return filtered


def metrics_for_performance_window(result_df):
    if result_df is None or result_df.empty:
        return {}, {}
    benchmark_ret = result_df["Benchmark_Return"] if "Benchmark_Return" in result_df.columns else pd.Series(dtype=float)
    strategy_metrics = performance_metrics(result_df["Strategy_Return"], benchmark_ret, RISK_FREE_ANNUAL) if "Strategy_Return" in result_df.columns else {}
    hedged_metrics = performance_metrics(result_df["Hedged_Strategy_Return"], benchmark_ret, RISK_FREE_ANNUAL) if "Hedged_Strategy_Return" in result_df.columns else {}
    strategy_metrics["Hedged_CAGR"] = hedged_metrics.get("CAGR")
    strategy_metrics["Hedged_Volatility"] = hedged_metrics.get("Volatility")
    strategy_metrics["Hedged_Sharpe_1pct"] = hedged_metrics.get("Sharpe_1pct")
    strategy_metrics["Hedged_Information_Ratio"] = hedged_metrics.get("Information_Ratio")
    strategy_metrics["Hedged_MaxDD"] = hedged_metrics.get("MaxDD")
    strategy_metrics["Hedged_Final_Equity"] = hedged_metrics.get("Final_Equity")
    return strategy_metrics, compute_benchmark_metrics_from_result(result_df)


def compute_benchmark_metrics_from_result(result_df):
    if result_df is None or result_df.empty or "Benchmark_Equity" not in result_df.columns:
        return {"Bench_CAGR": np.nan, "Bench_MaxDD": np.nan, "Bench_Sharpe": np.nan}
    eq = result_df["Benchmark_Equity"].dropna()
    if len(eq) < 3:
        return {"Bench_CAGR": np.nan, "Bench_MaxDD": np.nan, "Bench_Sharpe": np.nan}
    rets = eq.pct_change().dropna()
    years = max((eq.index[-1] - eq.index[0]).days / 365.25, 1e-9)
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / years) - 1
    dd = eq / eq.cummax() - 1
    sharpe = np.sqrt(52) * rets.mean() / rets.std() if rets.std() and np.isfinite(rets.std()) else np.nan
    return {"Bench_CAGR": cagr, "Bench_MaxDD": dd.min(), "Bench_Sharpe": sharpe}


def listing_country_from_ticker(ticker, fallback="-"):
    txt = safe_text(ticker, "").upper()
    suffix = txt.rsplit(".", 1)[1] if "." in txt else ""
    suffix_map = {
        "AS": "Netherlands",
        "BR": "Belgium",
        "CO": "Denmark",
        "DE": "Germany",
        "F": "Germany",
        "HE": "Finland",
        "IR": "Ireland",
        "L": "United Kingdom",
        "LS": "Portugal",
        "MC": "Spain",
        "MI": "Italy",
        "PA": "France",
        "SW": "Switzerland",
        "VI": "Austria",
        "OL": "Norway",
        "ST": "Sweden",
        "WA": "Poland",
        "PR": "Czech Republic",
        "AT": "Greece",
        "IS": "Turkey",
        "TW": "Taiwan",
        "TWO": "Taiwan",
        "KS": "South Korea",
        "KQ": "South Korea",
        "HK": "Hong Kong",
        "SS": "China",
        "SZ": "China",
        "NS": "India",
        "BO": "India",
        "SI": "Singapore",
        "JK": "Indonesia",
        "KL": "Malaysia",
        "BK": "Thailand",
        "SA": "Brazil",
        "MX": "Mexico",
        "JO": "South Africa",
        "TA": "Israel",
        "AX": "Australia",
        "NZ": "New Zealand",
        "TO": "Canada",
        "V": "Canada",
    }
    if suffix in suffix_map:
        return suffix_map[suffix]
    if txt.startswith("^"):
        return fallback
    return "United States" if suffix == "" else fallback


def fetch_fundamentals(tickers):
    rows = []
    for ticker in tickers:
        info = {}
        try:
            info = yf.Ticker(str(ticker)).get_info() or {}
        except Exception:
            info = {}
        rows.append({
            "Ticker": str(ticker),
            "Name": info.get("shortName") or info.get("longName") or str(ticker),
            "Country": listing_country_from_ticker(ticker, info.get("country") or "-"),
            "Sector": info.get("sector") or "-",
            "Industry": info.get("industry") or "-",
            "MarketCap": info.get("marketCap", np.nan),
            "LongBusinessSummary": info.get("longBusinessSummary") or "",
        })
    return pd.DataFrame(rows)


def read_dynamic_allocation_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        return {"available": False, "summary": "Dynamic allocation PDF not found.", "allocations": []}
    try:
        from pypdf import PdfReader
        text = "\n".join((page.extract_text() or "") for page in PdfReader(str(pdf_path)).pages)
    except Exception as exc:
        return {"available": False, "summary": f"Dynamic allocation PDF could not be read: {exc}", "allocations": []}
    import re
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    week_ended = "2026-06-12"
    for line in lines:
        m = re.search(r"Allocation week ended:\s*([0-9]{4}-[0-9]{2}-[0-9]{2})", line)
        if m:
            week_ended = m.group(1)
            break
    allocations = []
    pattern = re.compile(r"^(US100|Europe|Italy|GOLD|SPY|EXU|EZU|EWI|GLD|CASH)\s+([A-Z0-9.\-^=]+)\s+([+-]?[0-9]+(?:\.[0-9]+)?)%\s+([+-]?[0-9]+(?:\.[0-9]+)?)%\s+([+-]?[0-9]+(?:\.[0-9]+)?)%$")
    for line in lines:
        m = pattern.match(line)
        if m:
            allocations.append({
                "Asset": normalize_dynamic_asset_label(m.group(1)),
                "Ticker": m.group(2),
                "Weight": float(m.group(3)) / 100,
                "Previous_Weight": float(m.group(4)) / 100,
                "Change": float(m.group(5)) / 100,
            })
    if not allocations:
        tokens = [t for t in lines if t not in {"Asset", "Ticker", "Current", "Previous", "Change", "ASSET ALLOCATION"}]
        asset_names = {"US100", "Europe", "Italy", "GOLD", "SPY", "EXU", "EZU", "EQI", "GLD", "CASH"}
        i = 0
        while i <= len(tokens) - 5:
            asset, ticker = tokens[i], tokens[i + 1]
            vals = tokens[i + 2:i + 5]
            if asset in asset_names and all(re.match(r"^[+-]?[0-9]+(?:\.[0-9]+)?%$", v) for v in vals):
                allocations.append({
                    "Asset": normalize_dynamic_asset_label(asset),
                    "Ticker": ticker,
                    "Weight": float(vals[0].replace("%", "")) / 100,
                    "Previous_Weight": float(vals[1].replace("%", "")) / 100,
                    "Change": float(vals[2].replace("%", "")) / 100,
                })
                i += 5
            else:
                i += 1
    if allocations:
        allocation_text = ", ".join(f"{a['Asset']} {a['Weight']:.2%}" for a in allocations)
        changes = [a for a in allocations if abs(float(a.get("Change", 0.0))) > 1e-8]
        change_text = ", ".join(f"{a['Asset']} {a['Change']:+.2%}" for a in changes) if changes else "no material week-over-week changes"
        summary = f"Dynamic Risk Allocation for week ended {week_ended}: {allocation_text}. Week-over-week changes: {change_text}."
    else:
        summary = "Dynamic allocation PDF was read, but no allocation rows were detected."
    return {"available": bool(allocations), "summary": summary, "allocations": allocations, "week_ended": week_ended, "source": str(pdf_path)}


def make_summary_equity_drawdown_chart(result_df, market, benchmark, out_dir):
    import matplotlib.dates as mdates
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"summary_{safe_report_market_name(market)}.png"
    plot_df = result_df.copy().dropna(how="all")
    if "Exposure" in plot_df.columns and (plot_df["Exposure"].fillna(0) > 0).any():
        first_active = plot_df.index[plot_df["Exposure"].fillna(0) > 0][0]
        first_pos = max(0, plot_df.index.get_loc(first_active) - 1)
        plot_df = plot_df.iloc[first_pos:]
    elif "Strategy_Equity" in plot_df.columns:
        changed = plot_df["Strategy_Equity"].pct_change().abs().fillna(0) > 1e-10
        if changed.any():
            first_active = changed[changed].index[0]
            first_pos = max(0, plot_df.index.get_loc(first_active) - 1)
            plot_df = plot_df.iloc[first_pos:]
    equity_cols = [c for c in ["Strategy_Equity", "Hedged_Strategy_Equity", "Benchmark_Equity"] if c in plot_df.columns]
    for col in equity_cols:
        valid = plot_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if not valid.empty and valid.iloc[0] != 0:
            plot_df[col] = plot_df[col] / valid.iloc[0]

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(7.45, 5.30), sharex=True, gridspec_kw={"height_ratios": [2.90, 0.90, 0.82]})
    ax1.set_title("Equity", color="black", fontsize=9.2, fontweight="bold", pad=2)
    ax1.plot(plot_df.index, plot_df["Strategy_Equity"], color="#123E73", lw=1.3, label="Long")
    if "Hedged_Strategy_Equity" in plot_df:
        ax1.plot(plot_df.index, plot_df["Hedged_Strategy_Equity"], color="#008C7A", lw=1.2, label="Long+Hedge")
    if "Benchmark_Equity" in plot_df:
        ax1.plot(plot_df.index, plot_df["Benchmark_Equity"], color="#E31A1C", lw=1.1, ls="--", label="Benchmark")
    if equity_cols:
        equity_values = plot_df[equity_cols].replace([np.inf, -np.inf], np.nan).stack().dropna()
        if not equity_values.empty:
            ymin = float(equity_values.min())
            ymax = float(equity_values.max())
            margin = max((ymax - ymin) * 0.08, 0.03)
            ax1.set_ylim(ymin - margin, ymax + margin)
    ax1.legend(loc="upper center", ncol=3, fontsize=9.2, frameon=False)
    ax1.grid(True, alpha=0.18)
    ax1.tick_params(labelsize=8)
    ax2.set_title("Drawdown", color="black", fontsize=9.2, fontweight="bold", pad=2)
    if "Strategy_Drawdown" in result_df:
        ax2.plot(plot_df.index, plot_df["Strategy_Drawdown"], color="#123E73", lw=1.0)
    if "Hedged_Strategy_Drawdown" in result_df:
        ax2.plot(plot_df.index, plot_df["Hedged_Strategy_Drawdown"], color="#008C7A", lw=0.95)
    if "Benchmark_Drawdown" in result_df:
        ax2.plot(plot_df.index, plot_df["Benchmark_Drawdown"], color="#E31A1C", lw=0.9)
    ax2.grid(True, alpha=0.18)
    ax2.tick_params(labelsize=8)
    ax3.set_title("Exposure", color="black", fontsize=9.2, fontweight="bold", pad=18)
    exposure_series = []
    if "Exposure" in result_df:
        y = plot_df["Exposure"] * 100.0
        exposure_series.append(y)
        ax3.plot(plot_df.index, y, color="#123E73", lw=1.15, label="Long exp.")
    if "Hedge_Short_Target" in result_df:
        y = plot_df["Hedge_Short_Target"] * 100.0
        exposure_series.append(y)
        ax3.plot(plot_df.index, y, color="#D97706", lw=1.05, ls="--", label="Short hedge")
    if "Net_Exposure_After_Hedge" in result_df:
        y = plot_df["Net_Exposure_After_Hedge"] * 100.0
        exposure_series.append(y)
        ax3.plot(plot_df.index, y, color="#008C7A", lw=1.05, label="Net exp.")
    if exposure_series:
        exposure_values = pd.concat(exposure_series).replace([np.inf, -np.inf], np.nan).dropna()
        if not exposure_values.empty:
            ymin = float(exposure_values.min())
            ymax = float(exposure_values.max())
            margin = max((ymax - ymin) * 0.16, 2.5)
            ax3.set_ylim(ymin - margin, ymax + margin)
    ax3.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=8.8, frameon=False, borderaxespad=0.0)
    ax3.set_ylabel("Exp. %", fontsize=6)
    ax3.grid(True, alpha=0.18)
    ax3.tick_params(axis="both", labelsize=9)
    quarter_ticks = pd.date_range(plot_df.index.min(), plot_df.index.max(), freq="QS-DEC")
    quarter_ticks = quarter_ticks[(quarter_ticks >= plot_df.index.min()) & (quarter_ticks <= plot_df.index.max())]
    if len(quarter_ticks) > 6:
        tick_idx = np.linspace(0, len(quarter_ticks) - 1, 6).round().astype(int)
        quarter_ticks = quarter_ticks[tick_idx]
    ax3.set_xticks(quarter_ticks.to_pydatetime())
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    fig.autofmt_xdate(rotation=0, ha="center")
    fig.tight_layout(pad=0.32, h_pad=0.55)
    fig.savefig(out, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out


def _score_color(score):
    try:
        v = float(score)
    except Exception:
        v = 0
    if v >= 70:
        return "#00C853"
    if v >= 45:
        return "#FF9800"
    return "#FF2D2D"


def _draw_gradient_bar(ax, x, y, w, h, score):
    colors_list = ["#FF2D2D", "#FF9800", "#FFE600", "#00C853"]
    for i, c in enumerate(colors_list):
        ax.add_patch(plt.Rectangle((x + w * i / 4, y), w / 4, h, color=c, transform=ax.transAxes, clip_on=False))
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="#1F2933", linewidth=0.8, transform=ax.transAxes, clip_on=False))
    xpos = x + w * max(0, min(100, float(score))) / 100
    ax.plot([xpos, xpos], [y - 0.012, y + h + 0.012], color="#DCE8F5", lw=0.8, transform=ax.transAxes, clip_on=False)


def make_asset_detail_dashboard_page(ticker, market, benchmark, ohlc, score_history, detail_row, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{safe_report_market_name(ticker)}_asset_dashboard.png"
    df = ohlc.copy().dropna()
    if df.empty:
        raise ValueError(f"No OHLC data for {ticker}")
    df.index = pd.to_datetime(df.index).tz_localize(None)
    score_history = score_history.reindex(df.index).ffill().bfill() if score_history is not None and not score_history.empty else pd.DataFrame(index=df.index)
    trend_score = _safe_float(detail_row.get("Trend_Score"), _safe_float(score_history.get("TrendScore", pd.Series([np.nan])).iloc[-1] if "TrendScore" in score_history else np.nan, 0))
    stability_score = _safe_float(detail_row.get("Stability_Score"), _safe_float(score_history.get("StabilityScore", pd.Series([np.nan])).iloc[-1] if "StabilityScore" in score_history else np.nan, 0))
    selected = bool(detail_row.get("Selected", False))
    candidate = bool(detail_row.get("Candidate", False))
    current_status = "Selected" if selected else ("Watchlist" if candidate else "Rejected")
    status_color = {"Selected": "#00C853", "Watchlist": "#FF9800", "Rejected": "#FF2D2D"}.get(current_status, "#1F2933")
    name = safe_text(detail_row.get("Name"), ticker)
    country = safe_text(detail_row.get("Country"), market)
    sector = safe_text(detail_row.get("Sector"), "-")
    industry = safe_text(detail_row.get("Industry"), "-")
    report_date = pd.Timestamp(LAST_AVAILABLE_FRIDAY).strftime("%d %b %Y")

    fig = plt.figure(figsize=(14.0, 10.0), facecolor="white")
    ink = "#1F2933"
    blue = "#123E73"
    chart_bg = "#FBFCFE"
    panel_bg = "#F4F7FA"
    fig.text(0.035, 0.955, str(ticker), fontsize=34, fontweight="bold", color=blue, ha="left", va="top")
    fig.text(0.035, 0.890, name, fontsize=15, fontweight="bold", color=ink, ha="left")
    fig.text(0.410, 0.940, f"  {current_status.upper()}  ", fontsize=22, fontweight="bold", color=status_color, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.36", fc="white", ec=status_color, lw=1.8))
    fig.text(0.725, 0.932, f"Week ending: {report_date}", fontsize=12.5, color=ink, ha="left", va="center")

    ax_price = fig.add_axes([0.035, 0.285, 0.660, 0.555], facecolor=chart_bg)
    ax_trend = fig.add_axes([0.035, 0.205, 0.660, 0.05], facecolor=chart_bg, sharex=ax_price)
    ax_stability = fig.add_axes([0.035, 0.130, 0.660, 0.05], facecolor=chart_bg, sharex=ax_price)
    ax_status = fig.add_axes([0.035, 0.055, 0.660, 0.05], facecolor=chart_bg, sharex=ax_price)

    x = np.arange(len(df))
    for i, (_, row) in enumerate(df.iterrows()):
        o, h, l, c = [float(row.get(k, np.nan)) for k in ["Open", "High", "Low", "Close"]]
        if not all(np.isfinite([o, h, l, c])):
            continue
        col = "#12A65A" if c >= o else "#D7191C"
        ax_price.vlines(i, l, h, color=col, linewidth=0.6)
        ax_price.add_patch(plt.Rectangle((i - 0.32, min(o, c)), 0.64, max(abs(c - o), 1e-6), facecolor=col, edgecolor=col, linewidth=0.5))
    close = df["Close"].astype(float).dropna()
    reg_prices = close[close.index >= close.index.max() - pd.Timedelta(weeks=52)]
    if len(reg_prices) >= 8:
        rx = np.arange(len(reg_prices), dtype=float)
        y = np.log(reg_prices.values)
        slope, intercept = np.polyfit(rx, y, 1)
        fit = intercept + slope * rx
        resid = y - fit
        reg_std = np.nanstd(resid)
        full_x = np.arange(len(df))
        start_i = df.index.get_indexer([reg_prices.index[0]], method="nearest")[0]
        xx = np.arange(start_i, len(df))
        rel = xx - start_i
        center = np.exp(intercept + slope * rel)
        upper = np.exp(intercept + slope * rel + 2 * reg_std)
        lower = np.exp(intercept + slope * rel - 2 * reg_std)
        ax_price.plot(xx, center, color="#2E5FA8", lw=1.2, ls="--")
        ax_price.plot(xx, upper, color="#879BB5", lw=0.9, ls=":")
        ax_price.plot(xx, lower, color="#879BB5", lw=0.9, ls=":")
        ax_price.fill_between(xx, lower, upper, color="#AEBED1", alpha=0.16)
    ax_price.grid(True, color="#DCE5EF", linestyle="--", linewidth=0.45)
    ax_price.yaxis.tick_right()
    ax_price.tick_params(axis="both", labelsize=9, colors=ink)
    ax_price.tick_params(axis="x", labelbottom=False)
    for sp in ax_price.spines.values():
        sp.set_color("#C7D2DF")

    def bar_axis(ax, values, label):
        vals = pd.Series(values, index=df.index).reindex(df.index).ffill().bfill().fillna(0).clip(0, 100).values
        ax.bar(x, np.full(len(vals), 100.0), width=0.86, color=[_score_color(v) for v in vals], edgecolor="none")
        ax.set_ylim(0, 100)
        ax.yaxis.tick_right()
        ax.tick_params(axis="y", labelsize=8, colors="#536273")
        ax.tick_params(axis="x", labelbottom=False)
        ax.grid(False)
        ax.text(0.0, 1.12, label, transform=ax.transAxes, color=ink, fontsize=8, fontweight="bold", va="bottom", ha="left", clip_on=False)
        for sp in ax.spines.values():
            sp.set_color("#C7D2DF")
    bar_axis(ax_trend, score_history["TrendScore"] if "TrendScore" in score_history else pd.Series(trend_score, index=df.index), "Trend Score")
    bar_axis(ax_stability, score_history["StabilityScore"] if "StabilityScore" in score_history else pd.Series(stability_score, index=df.index), "Stability Score")
    status_vals = score_history["Status"] if "Status" in score_history else pd.Series(2 if selected else (1 if candidate else 0), index=df.index)
    ax_status.bar(x, np.full(len(status_vals), 100.0), width=0.86, color=["#00C853" if v >= 2 else ("#FF9800" if v >= 1 else "#FF2D2D") for v in status_vals], edgecolor="none")
    ax_status.set_ylim(0, 100)
    ax_status.yaxis.tick_right()
    ax_status.tick_params(axis="y", labelsize=8, colors="#536273")
    ax_status.text(0.0, 1.12, "Status", transform=ax_status.transAxes, color=ink, fontsize=8, fontweight="bold", va="bottom", ha="left", clip_on=False)
    tick_count = min(6, len(df))
    tick_locs = np.linspace(0, len(df) - 1, tick_count).astype(int)
    ax_status.set_xticks(tick_locs)
    ax_status.set_xticklabels([df.index[i].strftime("%b '%y") for i in tick_locs], fontsize=9, color=ink)
    for sp in ax_status.spines.values():
        sp.set_color("#C7D2DF")

    panel = fig.add_axes([0.725, 0.055, 0.23, 0.785], facecolor=panel_bg)
    panel.set_xticks([]); panel.set_yticks([])
    panel.set_xlim(0, 1); panel.set_ylim(0, 1)
    for sp in panel.spines.values(): sp.set_color("#C7D2DF")
    detail_items = [("Ticker", ticker), ("Name", name), ("Country", country), ("Sector", sector), ("Industry", industry), ("Benchmark", benchmark)]
    y = 0.92
    for label, value in detail_items:
        panel.text(0.05, y, label, color=ink, fontsize=10, va="center")
        panel.text(0.95, y, safe_text(value), color=ink, fontsize=10, va="center", ha="right")
        panel.plot([0.05, 0.95], [y - 0.055, y - 0.055], color="#D7E0EA", lw=0.7, transform=panel.transAxes)
        y -= 0.072
    panel.plot([0, 1], [0.55, 0.55], color="#C7D2DF", lw=0.8, transform=panel.transAxes)
    panel.text(0.05, 0.510, "SCORES", color=ink, fontsize=13, fontweight="bold", va="top")
    panel.text(0.05, 0.445, "Trend Score", color=ink, fontsize=10, va="center")
    panel.text(0.95, 0.445, f"{trend_score:.0f} / 100", color="#0087B7", fontsize=10, va="center", ha="right")
    _draw_gradient_bar(panel, 0.05, 0.400, 0.816, 0.012, trend_score)
    panel.text(0.05, 0.335, "Stability Score", color=ink, fontsize=10, va="center")
    panel.text(0.95, 0.335, f"{stability_score:.0f} / 100", color="#C56A00", fontsize=10, va="center", ha="right")
    _draw_gradient_bar(panel, 0.05, 0.290, 0.816, 0.012, stability_score)
    panel.plot([0, 1], [0.235, 0.235], color="#C7D2DF", lw=0.8, transform=panel.transAxes)
    panel.text(0.50, 0.185, f"  {current_status.upper()}  ", color=status_color, fontsize=15, fontweight="bold", ha="center", va="center", bbox=dict(boxstyle="round,pad=0.34", fc="white", ec=status_color, lw=1.3))
    panel.text(0.05, 0.105, "Position Size", color=ink, fontsize=10, va="center")
    panel.text(0.95, 0.105, "20% Capital" if selected else "0% Capital", color=ink, fontsize=10, va="center", ha="right")
    panel.text(0.05, 0.055, "Next Review", color=ink, fontsize=10, va="center")
    panel.text(0.95, 0.055, (pd.Timestamp(LAST_AVAILABLE_FRIDAY) + pd.Timedelta(days=7)).strftime("%d %b %Y"), color=ink, fontsize=10, va="center", ha="right")

    panel.set_xlim(0, 1); panel.set_ylim(0, 1)

    logo = find_logo_file()
    if logo:
        footer_logo = fig.add_axes([0.035, 0.000, 0.1365, 0.0585])
        footer_logo.imshow(plt.imread(str(logo)))
        footer_logo.axis("off")
    fig.text(0.50, 0.028, "All rights reserved", fontsize=8, color="#536273", ha="center")
    fig.savefig(out, dpi=180)
    plt.close(fig)
    return out


def make_asset_detail_dashboard_pages(selected_market, benchmark, px_univ, best_run, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pages = []
    ranking_by_date = best_run.get("ranking_by_date", {}) if isinstance(best_run, dict) else {}
    latest_date = sorted(ranking_by_date.keys())[-1] if ranking_by_date else px_univ.index[-1]
    latest = ranking_by_date.get(latest_date, pd.DataFrame()).copy()
    fundamentals = pd.DataFrame()
    tickers = list(px_univ.columns)
    try:
        fundamentals = fetch_fundamentals(tickers)
    except Exception:
        fundamentals = pd.DataFrame({"Ticker": tickers, "Name": tickers})
    for ticker in tickers:
        close = px_univ[ticker].dropna()
        if len(close) < 10:
            continue
        ohlc = pd.DataFrame({"Open": close.shift(1).fillna(close), "High": close * 1.01, "Low": close * 0.99, "Close": close}, index=close.index)
        hist_rows = []
        for d, rnk in ranking_by_date.items():
            rr = rnk[rnk["Ticker"].astype(str) == str(ticker)] if not rnk.empty and "Ticker" in rnk else pd.DataFrame()
            if rr.empty:
                continue
            row = rr.iloc[0]
            trend_hist_score = _safe_float(row.get("Trend_Score"), np.nan)
            if not np.isfinite(trend_hist_score):
                trend_hist_score = _safe_float(row.get("TrendScore"), np.nan)
            if not np.isfinite(trend_hist_score):
                trend_hist_score = 70.0 if bool(row.get("Candidate", False)) else 35.0
            stability_hist_score = _safe_float(row.get("Stability_Score"), np.nan)
            if not np.isfinite(stability_hist_score):
                stability_hist_score = _safe_float(row.get("StabilityScore"), np.nan)
            if not np.isfinite(stability_hist_score):
                stability_hist_score = _safe_float(row.get("R2"), 0.0) * 100.0
            hist_rows.append({"Date": d, "TrendScore": max(0.0, min(100.0, trend_hist_score)), "StabilityScore": max(0.0, min(100.0, stability_hist_score)), "Status": 2 if bool(row.get("Selected", False)) else (1 if bool(row.get("Candidate", False)) else 0)})
        score_history = pd.DataFrame(hist_rows).set_index("Date") if hist_rows else pd.DataFrame(index=close.index)
        detail = latest[latest["Ticker"].astype(str) == str(ticker)].iloc[0].copy() if not latest.empty and "Ticker" in latest and str(ticker) in latest["Ticker"].astype(str).values else pd.Series({"Ticker": ticker})
        finfo = fundamentals[fundamentals["Ticker"].astype(str) == str(ticker)]
        if not finfo.empty:
            for col in finfo.columns:
                if col not in detail or safe_text(detail.get(col)) == "-":
                    detail[col] = finfo.iloc[0].get(col)
        detail["Ticker"] = ticker
        try:
            pages.append(make_asset_detail_dashboard_page(ticker, selected_market, benchmark, ohlc, score_history, detail, out_dir))
        except Exception as exc:
            print(f"Asset detail skipped for {ticker}: {exc}")
    return pages


def _make_styles():
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="BatchTitle", parent=styles["Title"], fontSize=18, leading=22, textColor=colors.HexColor("#1F4E79"), alignment=1))
    styles.add(ParagraphStyle(name="BatchSection", parent=styles["Heading2"], fontSize=11.5, leading=14, textColor=colors.HexColor("#1F4E79"), spaceAfter=4))
    styles.add(ParagraphStyle(name="BatchSmall", parent=styles["Normal"], fontSize=7.8, leading=9.3))
    styles.add(ParagraphStyle(name="BatchFooter", parent=styles["Normal"], fontSize=6.6, leading=7.7))
    styles.add(ParagraphStyle(name="BatchTiny", parent=styles["Normal"], fontSize=7.4, leading=8.6))
    styles.add(ParagraphStyle(name="BatchTinyRight", parent=styles["BatchTiny"], alignment=2))
    styles.add(ParagraphStyle(name="BatchDisclaimer", parent=styles["Normal"], fontSize=5.3, leading=6.2))
    return styles


def read_report_text_file(filename, fallback="-"):
    path = BASE_DIR / filename
    try:
        text = path.read_text(encoding="utf-8").strip()
    except UnicodeDecodeError:
        text = path.read_text(encoding="latin-1").strip()
    except Exception:
        text = fallback
    return text or fallback


PROCESSING_OBJECTIVE_TEXT = read_report_text_file("ProcessingObjective.txt")
BIO_TEXT = read_report_text_file("Bio.txt")
DISCLAIMER_TEXT = read_report_text_file("Disclaimer.txt")


def generate_batch_summary_pdf(batch_summary_df):
    styles = _make_styles()
    ok_df = batch_summary_df.copy()
    market = safe_text(ok_df.iloc[0].get("Market"), "Preview") if not ok_df.empty else "Preview"
    out = OUTPUT_DIR / f"Summary_{safe_report_market_name(market)}.pdf"
    doc = SimpleDocTemplate(str(out), pagesize=landscape(A4), rightMargin=0.45 * cm, leftMargin=0.45 * cm, topMargin=0.50 * cm, bottomMargin=0.45 * cm)
    story = []
    logo = find_logo_file()
    header_cells = []
    if logo:
        header_cells.append(Image(str(logo), width=2.7 * cm, height=1.0 * cm))
    else:
        header_cells.append(Paragraph("TradingAlgo Mosaic", styles["BatchSmall"]))
    header_cells.append(Paragraph("TradingAlgo Mosaic Weekly Summary", styles["BatchTitle"]))
    header_cells.append(Paragraph(f"<b>Week ending:</b> {pd.Timestamp(LAST_AVAILABLE_FRIDAY).strftime('%d %b %Y')}", styles["BatchSmall"]))
    header = Table([header_cells], colWidths=[5.2 * cm, 16.2 * cm, 6.0 * cm])
    header.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "MIDDLE"), ("ALIGN", (1,0), (1,0), "CENTER"), ("ALIGN", (2,0), (2,0), "RIGHT"), ("LINEBELOW", (0,0), (-1,0), 0.8, colors.HexColor("#1F4E79"))]))
    story += [header, Spacer(1, 0.28 * cm)]
    dynamic_allocation = None
    if CREATE_DYNAMIC_RISK_ALLOCATION_TABLE:
        story += [Paragraph("Dynamic Risk Allocation", styles["BatchSection"]), Spacer(1, 0.08 * cm)]
        dynamic_allocation = read_dynamic_allocation_pdf(DYNAMIC_ALLOCATION_PDF_PATH)
        allocs = dynamic_allocation.get("allocations", [])
        if allocs:
            rows = [[Paragraph("<font color='white'><b>Asset</b></font>", styles["BatchTiny"]), Paragraph("<font color='white'><b>Ticker</b></font>", styles["BatchTiny"]), Paragraph("<font color='white'><b>Current</b></font>", styles["BatchTinyRight"]), Paragraph("<font color='white'><b>Previous</b></font>", styles["BatchTinyRight"]), Paragraph("<font color='white'><b>Change</b></font>", styles["BatchTinyRight"])]]
            for a in allocs:
                rows.append([Paragraph(pdf_text(normalize_dynamic_asset_label(a.get("Asset"))), styles["BatchTiny"]), Paragraph(pdf_text(a.get("Ticker")), styles["BatchTiny"]), Paragraph(format_pct_for_pdf(a.get("Weight")), styles["BatchTinyRight"]), Paragraph(format_pct_for_pdf(a.get("Previous_Weight")), styles["BatchTinyRight"]), Paragraph(format_pct_for_pdf(a.get("Change")), styles["BatchTinyRight"])])
            t = Table(rows, colWidths=[7.1*cm, 1.7*cm, 1.7*cm, 1.7*cm, 1.7*cm], hAlign="LEFT")
            allocation_style = [
                ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#1F4E79")),
                ("TEXTCOLOR", (0,0), (-1,0), colors.white),
                ("GRID", (0,0), (-1,-1), 0.25, colors.HexColor("#D0D0D0")),
                ("ALIGN", (2,1), (-1,-1), "RIGHT"),
                ("TOPPADDING", (0,0), (-1,-1), 2.2),
                ("BOTTOMPADDING", (0,0), (-1,-1), 2.2),
            ]
            current_asset_map = {"USA": "US100", "Europe": "Europe", "EmergingMarkets": "EmergingMarkets", "Emerging Markets": "EmergingMarkets", "Italy": "Italy"}
            current_asset = current_asset_map.get(market, market)
            for row_idx, allocation in enumerate(allocs, start=1):
                if normalize_dynamic_asset_label(allocation.get("Asset")) == current_asset:
                    allocation_style.extend([
                        ("BACKGROUND", (0,row_idx), (-1,row_idx), colors.HexColor("#DFF5E7")),
                        ("TEXTCOLOR", (0,row_idx), (-1,row_idx), colors.HexColor("#007A3D")),
                        ("FONTNAME", (0,row_idx), (-1,row_idx), "Helvetica-Bold"),
                    ])
                    break
            t.setStyle(TableStyle(allocation_style))
            story.append(t)
        story.append(Spacer(1, 0.14 * cm))
    story += [Paragraph("Portfolios", styles["BatchSection"]), Spacer(1, 0.12 * cm)]
    rows = [["Market", "Metrics", "Equity / Drawdown / Exposure", "Current Positions", "Weekly Changes"]]
    for _, r in ok_df.iterrows():
        metrics = Table([
            ["", "Long", "L+H", "B"],
            ["CAGR", format_pct_for_pdf(r.get("Strategy CAGR")), format_pct_for_pdf(r.get("Hedged CAGR")), format_pct_for_pdf(r.get("Bench Cagr"))],
            ["MaxDD", format_pct_for_pdf(r.get("Strategy MaxDD")), format_pct_for_pdf(r.get("Hedged MaxDD")), format_pct_for_pdf(r.get("Bench_MaxDD"))],
            ["Sharpe", format_num_for_pdf(r.get("Strategy Sharpe Ratio")), format_num_for_pdf(r.get("Hedged Sharpe Ratio")), format_num_for_pdf(r.get("Bench Sharpe"))],
            ["IR", format_num_for_pdf(r.get("Information Ratio")), format_num_for_pdf(r.get("Hedged Information Ratio")), "-"],
        ], colWidths=[1.18 * cm, 1.08 * cm, 1.08 * cm, 1.08 * cm], hAlign="CENTER")
        metrics.setStyle(TableStyle([
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTNAME", (0, 1), (0, -1), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (-1, -1), 7.4),
            ("ALIGN", (1, 0), (-1, -1), "RIGHT"),
            ("ALIGN", (0, 0), (0, -1), "LEFT"),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("LEFTPADDING", (0, 0), (-1, -1), 2.4),
            ("RIGHTPADDING", (0, 0), (-1, -1), 4.2),
            ("TOPPADDING", (0, 0), (-1, -1), 2.0),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 2.0),
        ]))
        chart = Image(str(r.get("Summary_Chart")), width=11.70*cm, height=7.35*cm) if r.get("Summary_Chart") and Path(str(r.get("Summary_Chart"))).exists() else Paragraph("-", styles["BatchTiny"])
        positions = Paragraph(format_current_positions_for_pdf(r.get("Last Weekly Selection")), styles["BatchTiny"])
        hedge_ticker = pdf_text(r.get("Benchmark Hedge Ticker", r.get("Benchmark")))
        changes = Paragraph(f"{format_pct_for_pdf(r.get('Capital Invested'))} Capital Invested<br/><br/><font color='green'><b>IN</b></font><br/>{pdf_text(r.get('Added Tickers'))}<br/><br/><font color='red'><b>OUT</b></font><br/>{pdf_text(r.get('Removed Tickers'))}<br/><br/><font color='#D97706'><b>HEDGE</b></font><br/>Short {hedge_ticker}: {format_pct_for_pdf(r.get('Benchmark Hedge Short'))}", styles["BatchTiny"])
        market_cell = Paragraph(f"<font size='9.6'><b>{pdf_text(r.get('Market'))}</b></font><br/><font size='6.6'>(benchmark {pdf_text(r.get('Benchmark'))})</font><br/><br/><font size='7.2'>{str(int(_safe_float(r.get('Number of Tickers'), 0)))} tickers</font>", styles["BatchTiny"])
        rows.append([market_cell, metrics, chart, positions, changes])
    col_widths = [2.60*cm, 4.80*cm, 12.10*cm, 3.60*cm, 4.80*cm]
    portfolio_row_heights = [None] + [8.05 * cm] * max(0, len(rows) - 1)
    pt = Table(rows, colWidths=col_widths, rowHeights=portfolio_row_heights, repeatRows=1, hAlign="CENTER")
    pt.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.HexColor("#1F4E79")), ("TEXTCOLOR", (0,0), (-1,0), colors.white), ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,0), 8), ("FONTNAME", (0,1), (-1,-1), "Helvetica"), ("FONTSIZE", (0,1), (-1,-1), 7.6), ("FONTNAME", (0,1), (0,-1), "Helvetica-Bold"), ("ALIGN", (0,0), (-1,0), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"), ("GRID", (0,0), (-1,-1), 0.25, colors.HexColor("#D0D0D0")), ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#F2F6FA")]), ("TOPPADDING", (0,0), (-1,-1), 12.0), ("BOTTOMPADDING", (0,0), (-1,-1), 9.0), ("TOPPADDING", (1,1), (-1,-1), 13.0), ("BOTTOMPADDING", (1,1), (-1,-1), 10.0), ("TOPPADDING", (2,1), (2,-1), 2.0), ("BOTTOMPADDING", (2,1), (2,-1), 2.0), ("LEFTPADDING", (2,1), (2,-1), 2.0), ("RIGHTPADDING", (2,1), (2,-1), 2.0)]))
    story.append(pt)
    footer = Table([[ [Paragraph("Processing Objective", styles["BatchSection"]), Paragraph(pdf_text(PROCESSING_OBJECTIVE_TEXT), styles["BatchFooter"])], [Paragraph("Bio", styles["BatchSection"]), Paragraph(pdf_text(BIO_TEXT), styles["BatchFooter"])], [Paragraph("Disclaimer", styles["BatchSection"]), Paragraph(pdf_text(DISCLAIMER_TEXT), styles["BatchFooter"])] ]], colWidths=[8.55*cm, 8.55*cm, 8.55*cm])
    footer.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "TOP"), ("LINEBEFORE", (1,0), (1,0), 0.35, colors.HexColor("#C8C8C8")), ("LINEBEFORE", (2,0), (2,0), 0.35, colors.HexColor("#C8C8C8")), ("LEFTPADDING", (0,0), (-1,-1), 0), ("RIGHTPADDING", (0,0), (-1,-1), 8)]))
    story += [Spacer(1, 0.36 * cm), footer]
    for _, r in ok_df.iterrows():
        for page in r.get("Asset_Detail_Pages") if isinstance(r.get("Asset_Detail_Pages"), list) else []:
            if page and Path(str(page)).exists():
                story += [PageBreak(), Image(str(page), width=27.2*cm, height=19.0*cm)]
    doc.build(story)
    print("PDF summary batch creato:")
    print(out)
    if CREATE_DYNAMIC_RISK_ALLOCATION_TABLE and dynamic_allocation is not None:
        print("\nDYNAMIC RISK ALLOCATION")
        print(dynamic_allocation.get("summary", "-"))
    return out

def empty_market_summary_row(market_info, error):
    selected_market = market_info.get("Market", "-")
    benchmark = market_info.get("Benchmark", "-")
    universe = market_info.get("Universe", [])
    return {
        "Market": selected_market,
        "Benchmark": benchmark,
        "Number of Tickers": len(universe),
        "Strategy CAGR": np.nan,
        "Hedged CAGR": np.nan,
        "Strategy MaxDD": np.nan,
        "Hedged MaxDD": np.nan,
        "Strategy Sharpe Ratio": np.nan,
        "Hedged Sharpe Ratio": np.nan,
        "Bench Cagr": np.nan,
        "Bench_MaxDD": np.nan,
        "Bench Sharpe": np.nan,
        "Information Ratio": np.nan,
        "Hedged Information Ratio": np.nan,
        "PDF_Output": "",
        "Summary_Chart": "",
        "Capital Invested": np.nan,
        "Benchmark Hedge Short": np.nan,
        "Benchmark Hedge Ticker": benchmark,
        "Hedge Portfolio Score": np.nan,
        "Last Weekly Selection": "",
        "Added Tickers": "",
        "Removed Tickers": "",
        "Selection Drivers": "",
        "Strategy_Returns": {},
        "Benchmark_Returns": {},
        "Status_History": {},
        "Asset_Detail_Pages": [],
        "Report_PDF": "",
        "Summary_PDF_Temp": "",
        "Detailed_PDF_Temp": "",
        "Status": "ERROR",
        "Error": str(error),
    }


def merge_pdf_files(pdf_paths, output_path):
    from pypdf import PdfReader, PdfWriter

    writer = PdfWriter()
    for pdf_path in pdf_paths:
        pdf_path = Path(pdf_path)
        if not pdf_path.exists():
            continue
        reader = PdfReader(str(pdf_path))
        for page in reader.pages:
            writer.add_page(page)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as fh:
        writer.write(fh)
    return output_path


def remove_pdf_if_exists(pdf_path):
    try:
        pdf_path = Path(pdf_path)
        if pdf_path.exists():
            pdf_path.unlink()
    except Exception as e:
        print(f"PDF temporaneo non rimosso ({pdf_path}): {e}")


def cleanup_market_report_outputs(selected_market):
    safe_market = safe_report_market_name(selected_market)
    for prefix in ["Report_", "Summary_", "Detailed_"]:
        remove_pdf_if_exists(OUTPUT_DIR / f"{prefix}{safe_market}.pdf")


def final_market_report_path(selected_market):
    return OUTPUT_DIR / f"Report_{safe_report_market_name(selected_market)}.pdf"


def merge_pdf_files_with_fallback(pdf_parts, output_path):
    output_path = Path(output_path)
    try:
        merge_pdf_files(pdf_parts, output_path)
        return output_path
    except PermissionError:
        fallback_candidates = [
            output_path.with_name(f"{output_path.stem}_latest{output_path.suffix}"),
            output_path.with_name(f"{output_path.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}{output_path.suffix}"),
        ]
        for fallback_path in fallback_candidates:
            try:
                if fallback_path.exists():
                    fallback_path.unlink()
                merge_pdf_files(pdf_parts, fallback_path)
                print("Report target locked; saved updated report as:", fallback_path)
                return fallback_path
            except PermissionError:
                continue
        raise



def resolve_site_reports_dir():
    configured_dir = globals().get("SITE_REPORTS_DIR", None)
    if configured_dir:
        return Path(configured_dir)

    site_project_dir = Path(globals().get("SITE_PROJECT_DIR", BASE_DIR.parent / "TradingAlgo.it"))
    candidates = [
        site_project_dir / "public" / "reports",
        site_project_dir / "static" / "reports",
        site_project_dir / "assets" / "reports",
        site_project_dir / "reports",
    ]
    for candidate in candidates:
        if candidate.parent.exists() or candidate.exists():
            return candidate
    return candidates[0]



def publish_market_reports_to_site(market_reports_df):
    if not bool(globals().get("UPDATE_SITE_REPORTS", False)):
        print("Aggiornamento sito disabilitato: UPDATE_SITE_REPORTS=False")
        return None

    site_project_dir = Path(globals().get("SITE_PROJECT_DIR", BASE_DIR.parent / "TradingAlgo.it"))
    if not site_project_dir.exists():
        print("ATTENZIONE: progetto sito TradingAlgo.it non trovato:", site_project_dir)
        print("Creo comunque la cartella report configurata. Verifica che il percorso sia quello pubblicato dal sito.")

    site_reports_dir = resolve_site_reports_dir()
    site_reports_dir.mkdir(parents=True, exist_ok=True)

    import json as _json
    import shutil

    published_rows = []
    for _, row in market_reports_df.iterrows():
        if safe_text(row.get("Status")) != "OK":
            continue
        src = Path(str(row.get("Report_PDF", "")))
        if not src.exists():
            print(f"Report non trovato, sito non aggiornato per {row.get('Market')}: {src}")
            continue

        market = safe_text(row.get("Market"), "Report")
        filename = f"Report_{safe_report_market_name(market)}.pdf"
        dst = site_reports_dir / filename
        shutil.copy2(src, dst)

        stable_urls = globals().get("SITE_REPORT_URLS", {}) or {}
        published_rows.append({
            "Market": market,
            "Benchmark": safe_text(row.get("Benchmark")),
            "Status": safe_text(row.get("Status")),
            "Report_File": filename,
            "Report_Path": str(dst),
            "Report_URL": stable_urls.get(market, ""),
            "Updated_At": datetime.now(ZoneInfo("Europe/Rome")).isoformat(timespec="seconds"),
        })

    index_json = site_reports_dir / "reports_index.json"
    index_csv = site_reports_dir / "reports_index.csv"
    index_json.write_text(_json.dumps(published_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    pd.DataFrame(published_rows).to_csv(index_csv, index=False, encoding="utf-8")

    print("\nSITO AGGIORNATO")
    print("Cartella report sito:", site_reports_dir)
    print("Report pubblicati:", len(published_rows))
    print("Indice JSON:", index_json)
    print("Indice CSV:", index_csv)
    for row in published_rows:
        if row.get("Report_URL"):
            print(f"Link stabile {row['Market']}:", row["Report_URL"])
    return site_reports_dir



def run_independent_market_batch(market_info):
    selected_market = market_info["Market"]
    benchmark = market_info["Benchmark"]
    universe = market_info["Universe"]
    all_tickers = [benchmark] + universe

    print("\n" + "=" * 80)
    print(f"BATCH INDIPENDENTE: {selected_market}")
    print(f"Benchmark: {benchmark}")
    print(f"Ticker universo: {len(universe)}")
    print("=" * 80)
    cleanup_market_report_outputs(selected_market)

    try:
        prices_daily = download_prices_yfinance(
            all_tickers,
            START_DATE,
            END_DATE,
            benchmark_ticker=benchmark,
            market_name=selected_market,
        )
        print("Date range:", prices_daily.index.min().date(), "->", prices_daily.index.max().date())
        print("Colonne scaricate:", list(prices_daily.columns))

        px_univ, px_bench, ret_univ, ret_bench = prepare_weekly_prices(
            prices_daily,
            benchmark,
            universe,
            WEEKLY_RULE,
        )
        print("Ticker investibili disponibili:", list(px_univ.columns))
        print("Settimane:", len(px_univ))

        runs_df, all_runs, best_run = run_single_configuration_for_market(px_univ, ret_univ, ret_bench)

        detailed_pdf_out = ""

        full_result_for_summary = best_run["result"].copy()
        result_for_performance = filter_result_for_performance_window(full_result_for_summary, PERFORMANCE_START_DATE)
        strategy_metrics, benchmark_metrics = metrics_for_performance_window(result_for_performance)

        last_ranking = best_run["ranking_by_date"].get(px_univ.index[-1], pd.DataFrame()).copy()
        selected_tickers = []
        if not last_ranking.empty and "Selected" in last_ranking.columns and "Ticker" in last_ranking.columns:
            selected_tickers = last_ranking.loc[last_ranking["Selected"], "Ticker"].astype(str).tolist()

        previous_selected_tickers = []
        ranking_dates = sorted(best_run["ranking_by_date"].keys())
        if len(ranking_dates) >= 2:
            previous_ranking = best_run["ranking_by_date"].get(ranking_dates[-2], pd.DataFrame()).copy()
            if not previous_ranking.empty and "Selected" in previous_ranking.columns and "Ticker" in previous_ranking.columns:
                previous_selected_tickers = previous_ranking.loc[previous_ranking["Selected"], "Ticker"].astype(str).tolist()

        added_tickers_for_summary = [ticker for ticker in selected_tickers if ticker not in previous_selected_tickers]
        removed_tickers_for_summary = [ticker for ticker in previous_selected_tickers if ticker not in selected_tickers]

        selection_driver_lines = []
        if not last_ranking.empty and selected_tickers:
            selected_ranking = last_ranking[last_ranking["Ticker"].astype(str).isin(selected_tickers)].copy()
            for _, driver_row in selected_ranking.iterrows():
                ticker = safe_text(driver_row.get("Ticker"))
                selection_driver_lines.append(
                    f"{ticker}: trend slope {format_num_for_pdf(driver_row.get('Slope'))}, stability R2 {format_num_for_pdf(driver_row.get('R2'))}, score {format_num_for_pdf(driver_row.get('Score'))} and passed correlation filter"
                )

        current_selection_text = "-"
        if selected_tickers:
            try:
                selected_info = fetch_fundamentals(selected_tickers)
                selected_rows = []
                for _, selected_row in selected_info.iterrows():
                    selected_rows.append({
                        "Ticker": safe_text(selected_row.get("Ticker")),
                        "Name": safe_text(selected_row.get("Name")),
                        "Sector": safe_text(selected_row.get("Sector")),
                        "Industry": safe_text(selected_row.get("Industry")),
                    })
                current_selection_text = selected_rows
            except Exception:
                current_selection_text = "<br/>".join(selected_tickers)

        changed_tickers_for_summary = sorted(set(added_tickers_for_summary + removed_tickers_for_summary))
        changed_ticker_names = {}
        if changed_tickers_for_summary:
            try:
                changed_info = fetch_fundamentals(changed_tickers_for_summary)
                for _, changed_row in changed_info.iterrows():
                    changed_ticker_names[safe_text(changed_row.get("Ticker"))] = safe_text(changed_row.get("Name"))
            except Exception:
                changed_ticker_names = {}

        def format_changed_tickers(tickers):
            rows = []
            for ticker in tickers:
                name = changed_ticker_names.get(ticker, ticker)
                rows.append(f"{ticker} - {name}")
            return "<br/>".join(rows) if rows else "-"

        added_tickers_text = format_changed_tickers(added_tickers_for_summary)
        removed_tickers_text = format_changed_tickers(removed_tickers_for_summary)

        result_for_summary = result_for_performance.copy()
        profit_contributors_for_summary = []
        asset_profit_contributions = best_run.get("asset_profit_contributions", pd.DataFrame())
        if isinstance(asset_profit_contributions, pd.DataFrame) and not asset_profit_contributions.empty and not result_for_summary.empty:
            contribution_window = asset_profit_contributions.reindex(result_for_summary.index).fillna(0.0)
            start_idx = result_for_summary.index[0]
            full_equity_before = full_result_for_summary["Strategy_Equity"].shift(1).reindex([start_idx]).iloc[0] if "Strategy_Equity" in full_result_for_summary.columns else 1.0
            if not np.isfinite(full_equity_before) or full_equity_before == 0:
                full_equity_before = 1.0
            ticker_profit_contributors = (contribution_window.sum(axis=0) / float(full_equity_before)).replace([np.inf, -np.inf], np.nan).dropna().sort_values(ascending=False)
            for ticker, contribution in ticker_profit_contributors.items():
                profit_contributors_for_summary.append({
                    "Ticker": safe_text(ticker),
                    "Contribution": float(contribution),
                })
        current_capital_invested = result_for_summary["Exposure"].iloc[-1] if "Exposure" in result_for_summary.columns and not result_for_summary.empty else np.nan
        current_benchmark_hedge = result_for_summary["Hedge_Short_Target"].iloc[-1] if "Hedge_Short_Target" in result_for_summary.columns and not result_for_summary.empty else np.nan
        current_hedge_score = result_for_summary["Hedge_Portfolio_Score"].iloc[-1] if "Hedge_Portfolio_Score" in result_for_summary.columns and not result_for_summary.empty else np.nan
        strategy_returns_for_summary = result_for_summary["Strategy_Equity"].pct_change().dropna()
        hedged_returns_for_summary = result_for_summary["Hedged_Strategy_Equity"].pct_change().dropna() if "Hedged_Strategy_Equity" in result_for_summary.columns else pd.Series(dtype=float)
        benchmark_returns_for_summary = result_for_summary["Benchmark_Equity"].pct_change().dropna()

        summary_chart_out = make_summary_equity_drawdown_chart(
            result_for_summary,
            selected_market,
            benchmark,
            OUTPUT_DIR / "_summary_charts" / safe_report_market_name(selected_market),
        )

        status_dates = sorted(best_run["ranking_by_date"].keys())[-8:]
        status_tickers = []
        seen_status_tickers = set()
        for status_date in status_dates:
            status_ranking = best_run["ranking_by_date"].get(status_date, pd.DataFrame()).copy()
            if not status_ranking.empty and "Ticker" in status_ranking.columns:
                for status_ticker in status_ranking["Ticker"].astype(str).tolist():
                    if status_ticker not in seen_status_tickers:
                        status_tickers.append(status_ticker)
                        seen_status_tickers.add(status_ticker)
        latest_order = []
        if not last_ranking.empty and "Ticker" in last_ranking.columns:
            latest_order = last_ranking["Ticker"].astype(str).tolist()
        status_tickers = sorted(status_tickers, key=lambda ticker: latest_order.index(ticker) if ticker in latest_order else len(latest_order))
        status_history_rows = []
        for status_ticker in status_tickers:
            statuses = []
            for status_date in status_dates:
                status_ranking = best_run["ranking_by_date"].get(status_date, pd.DataFrame()).copy()
                status_value = "Rejected"
                if not status_ranking.empty and "Ticker" in status_ranking.columns:
                    ticker_rows = status_ranking[status_ranking["Ticker"].astype(str) == status_ticker]
                    if not ticker_rows.empty:
                        ticker_row = ticker_rows.iloc[0]
                        if bool(ticker_row.get("Selected", False)):
                            status_value = "Selected"
                        elif bool(ticker_row.get("Candidate", False)):
                            status_value = "Watchlist"
                statuses.append(status_value)
            status_history_rows.append({"Ticker": status_ticker, "Statuses": statuses})
        status_history = {
            "dates": [status_date.strftime("%m-%d") for status_date in status_dates],
            "rows": status_history_rows,
        }

        asset_detail_pages = make_asset_detail_dashboard_pages(
            selected_market,
            benchmark,
            px_univ,
            best_run,
            OUTPUT_DIR / "_asset_detail_pages" / safe_report_market_name(selected_market),
        )

        market_row = {
            "Market": selected_market,
            "Benchmark": benchmark,
            "Number of Tickers": len(px_univ.columns),
            "Strategy CAGR": strategy_metrics.get("CAGR"),
            "Hedged CAGR": strategy_metrics.get("Hedged_CAGR"),
            "Strategy MaxDD": strategy_metrics.get("MaxDD"),
            "Hedged MaxDD": strategy_metrics.get("Hedged_MaxDD"),
            "Strategy Sharpe Ratio": strategy_metrics.get("Sharpe_1pct"),
            "Hedged Sharpe Ratio": strategy_metrics.get("Hedged_Sharpe_1pct"),
            "Bench Cagr": benchmark_metrics.get("Bench_CAGR"),
            "Bench_MaxDD": benchmark_metrics.get("Bench_MaxDD"),
            "Bench Sharpe": benchmark_metrics.get("Bench_Sharpe"),
            "Information Ratio": strategy_metrics.get("Information_Ratio"),
            "Hedged Information Ratio": strategy_metrics.get("Hedged_Information_Ratio"),
            "PDF_Output": "",
            "Summary_Chart": str(summary_chart_out),
            "Capital Invested": current_capital_invested,
            "Benchmark Hedge Short": current_benchmark_hedge,
            "Benchmark Hedge Ticker": benchmark,
            "Hedge Portfolio Score": current_hedge_score,
            "Last Weekly Selection": current_selection_text,
            "Added Tickers": added_tickers_text,
            "Removed Tickers": removed_tickers_text,
            "Selection Drivers": "<br/>".join(selection_driver_lines) if selection_driver_lines else "-",
            "Profit_Contributors": profit_contributors_for_summary,
            "Strategy_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in strategy_returns_for_summary.items()},
            "Hedged_Strategy_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in hedged_returns_for_summary.items()},
            "Benchmark_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in benchmark_returns_for_summary.items()},
            "Status_History": status_history,
            "Asset_Detail_Pages": asset_detail_pages,
            "Report_PDF": "",
            "Summary_PDF_Temp": "",
            "Detailed_PDF_Temp": "",
            "Status": "OK",
            "Error": "",
        }

        single_market_df = pd.DataFrame([market_row])
        summary_pdf_out = generate_batch_summary_pdf(single_market_df)
        market_row["Summary_PDF_Temp"] = str(summary_pdf_out) if summary_pdf_out else ""

        final_pdf_out = final_market_report_path(selected_market)
        pdf_parts = [path for path in [summary_pdf_out] if path]
        if pdf_parts:
            final_pdf_out = merge_pdf_files_with_fallback(pdf_parts, final_pdf_out)
        market_row["Report_PDF"] = str(final_pdf_out) if final_pdf_out.exists() else ""

        remove_pdf_if_exists(summary_pdf_out)

        print("\nREPORT MERCATO COMPLETATO")
        print("Report PDF:", market_row["Report_PDF"])
        return market_row

    except Exception as e:
        print(f"ERRORE su {selected_market}: {e}")
        error_row = empty_market_summary_row(market_info, e)
        error_df = pd.DataFrame([error_row])
        try:
            summary_pdf_out = generate_batch_summary_pdf(error_df)
            error_row["Summary_PDF_Temp"] = str(summary_pdf_out) if summary_pdf_out else ""
            final_pdf_out = final_market_report_path(selected_market)
            if summary_pdf_out:
                final_pdf_out = merge_pdf_files_with_fallback([summary_pdf_out], final_pdf_out)
                error_row["Report_PDF"] = str(final_pdf_out) if final_pdf_out.exists() else ""
                remove_pdf_if_exists(summary_pdf_out)
        except Exception as summary_error:
            print(f"Summary PDF non generato per {selected_market}: {summary_error}")
        return error_row


def run_preview_graph_page():
    print("RUN_MODE=preview: genero Summary completo + una pagina grafica senza download dati.")
    preview_dir = OUTPUT_DIR / "preview"
    preview_dir.mkdir(parents=True, exist_ok=True)

    idx = pd.date_range(end=pd.Timestamp(LAST_AVAILABLE_FRIDAY), periods=90, freq="W-FRI")
    rng = np.random.default_rng(42)
    close = 100.0 * np.exp(np.cumsum(rng.normal(0.004, 0.035, len(idx))))
    open_px = close * (1.0 + rng.normal(0.0, 0.012, len(idx)))
    high = np.maximum(open_px, close) * (1.0 + rng.uniform(0.004, 0.028, len(idx)))
    low = np.minimum(open_px, close) * (1.0 - rng.uniform(0.004, 0.028, len(idx)))
    ohlc = pd.DataFrame({"Open": open_px, "High": high, "Low": low, "Close": close}, index=idx)

    trend = np.clip(np.linspace(35, 92, len(idx)) + rng.normal(0, 7, len(idx)), 0, 100)
    stability = np.clip(np.linspace(45, 84, len(idx)) + rng.normal(0, 6, len(idx)), 0, 100)
    status = np.where(trend > 70, 2.0, np.where(trend > 52, 1.0, 0.0))
    score_history = pd.DataFrame({
        "TrendScore": trend,
        "StabilityScore": stability,
        "Status": status,
    }, index=idx)

    detail_row = pd.Series({
        "Ticker": "PREVIEW",
        "Name": "Preview Asset Name",
        "Country": "United States",
        "Trend_Score": float(trend[-1]),
        "Stability_Score": float(stability[-1]),
        "Portfolio_Contribution_Score": 80.0,
        "Slope": 0.018,
        "R2": 0.72,
        "Score": 0.013,
        "Selected": True,
        "Candidate": True,
        "Sector": "Preview",
        "Industry": "Layout validation",
        "MarketCap": np.nan,
        "LongBusinessSummary": "Synthetic preview page used to validate chart layout without downloading market data.",
    })

    preview_path = make_asset_detail_dashboard_page(
        "PREVIEW",
        "Preview",
        "BENCH",
        ohlc,
        score_history,
        detail_row,
        preview_dir,
    )

    result_df = pd.DataFrame({
        "Strategy_Equity": close / close[0],
        "Benchmark_Equity": (close / close[0]) * np.linspace(1.0, 0.94, len(idx)),
    }, index=idx)
    result_df["Hedged_Strategy_Equity"] = result_df["Strategy_Equity"] * np.linspace(1.0, 1.035, len(idx))
    result_df["Exposure"] = 1.0
    result_df["Hedge_Short_Target"] = 0.20
    result_df["Net_Exposure_After_Hedge"] = 0.80
    result_df["Strategy_Drawdown"] = result_df["Strategy_Equity"] / result_df["Strategy_Equity"].cummax() - 1.0
    result_df["Hedged_Strategy_Drawdown"] = result_df["Hedged_Strategy_Equity"] / result_df["Hedged_Strategy_Equity"].cummax() - 1.0
    result_df["Benchmark_Drawdown"] = result_df["Benchmark_Equity"] / result_df["Benchmark_Equity"].cummax() - 1.0
    summary_chart_out = make_summary_equity_drawdown_chart(result_df, "Preview", "BENCH", preview_dir / "summary_charts")

    status_dates = list(idx[-8:])
    status_history = {
        "dates": [d.strftime("%m-%d") for d in status_dates],
        "rows": [
            {"Ticker": "PREVIEW", "Statuses": ["Watchlist"] * 3 + ["Selected"] * 5},
            {"Ticker": "SAMPLE", "Statuses": ["Rejected", "Watchlist", "Watchlist", "Selected", "Selected", "Selected", "Watchlist", "Selected"]},
        ],
    }
    strategy_returns = result_df["Strategy_Equity"].pct_change().dropna()
    hedged_returns = result_df["Hedged_Strategy_Equity"].pct_change().dropna()
    benchmark_returns = result_df["Benchmark_Equity"].pct_change().dropna()
    preview_row = {
        "Market": "Preview",
        "Benchmark": "BENCH",
        "Number of Tickers": 49,
        "Strategy CAGR": 0.1234,
        "Hedged CAGR": 0.1412,
        "Strategy MaxDD": float(result_df["Strategy_Drawdown"].min()),
        "Hedged MaxDD": float(result_df["Hedged_Strategy_Drawdown"].min()),
        "Strategy Sharpe Ratio": 1.23,
        "Hedged Sharpe Ratio": 1.36,
        "Bench Cagr": 0.0876,
        "Bench_MaxDD": float(result_df["Benchmark_Drawdown"].min()),
        "Bench Sharpe": 0.98,
        "Information Ratio": 0.42,
        "Hedged Information Ratio": 0.55,
        "PDF_Output": "",
        "Summary_Chart": str(summary_chart_out),
        "Capital Invested": 1.0,
        "Benchmark Hedge Short": 0.20,
        "Benchmark Hedge Ticker": "BENCH",
        "Hedge Portfolio Score": 80.0,
        "Last Weekly Selection": [
            {"Ticker": "PREVIEW", "Name": "Preview Asset Name", "Sector": "Preview", "Industry": "Layout validation"},
            {"Ticker": "SAMPLE", "Name": "Placeholder Holding", "Sector": "Sample sector", "Industry": "Synthetic component"},
            {"Ticker": "TEST", "Name": "Synthetic Component", "Sector": "Testing", "Industry": "Report preview"},
        ],
        "Added Tickers": "PREVIEW - Preview Asset Name",
        "Removed Tickers": "OLD - Previous Placeholder",
        "Selection Drivers": "PREVIEW: trend slope 0.02, stability R2 0.72, score 0.013 and passed correlation filter",
        "Strategy_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in strategy_returns.items()},
        "Hedged_Strategy_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in hedged_returns.items()},
        "Benchmark_Returns": {idx.strftime("%Y-%m-%d"): float(val) for idx, val in benchmark_returns.items()},
        "Status_History": status_history,
        "Asset_Detail_Pages": [],
        "Report_PDF": "",
        "Summary_PDF_Temp": "",
        "Detailed_PDF_Temp": "",
        "Status": "OK",
        "Error": "",
    }

    preview_summary_pdf = generate_batch_summary_pdf(pd.DataFrame([preview_row]))
    preview_pdf = preview_dir / "Preview_Report.pdf"
    summary_only_pdf = preview_dir / "Preview_Report_summary_only.pdf"
    merge_pdf_files([preview_summary_pdf], summary_only_pdf)
    remove_pdf_if_exists(preview_summary_pdf)

    # Append the graph page to the full summary preview PDF.
    graph_pdf = preview_dir / "Preview_Graph_Page.pdf"
    story = [Image(str(preview_path), width=27.2 * cm, height=19.0 * cm)]
    doc = SimpleDocTemplate(
        str(graph_pdf),
        pagesize=landscape(A4),
        rightMargin=0.55 * cm,
        leftMargin=0.55 * cm,
        topMargin=0.55 * cm,
        bottomMargin=0.55 * cm,
    )
    doc.build(story)
    combined_pdf = preview_dir / "Preview_Report_combined.pdf"
    merge_pdf_files([summary_only_pdf, graph_pdf], combined_pdf)
    remove_pdf_if_exists(summary_only_pdf)
    remove_pdf_if_exists(graph_pdf)
    final_preview_pdf = preview_pdf
    try:
        if preview_pdf.exists():
            preview_pdf.unlink()
        combined_pdf.replace(preview_pdf)
    except PermissionError:
        fallback_candidates = [
            preview_dir / "Preview_Report_latest.pdf",
            preview_dir / f"Preview_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf",
        ]
        for fallback_pdf in fallback_candidates:
            try:
                if fallback_pdf.exists():
                    fallback_pdf.unlink()
                combined_pdf.replace(fallback_pdf)
                final_preview_pdf = fallback_pdf
                print("Preview report target is locked; saved updated preview as:", final_preview_pdf)
                break
            except PermissionError:
                continue
        else:
            raise

    print("Preview graph page:", preview_path)
    print("Preview PDF:", final_preview_pdf)
    try:
        display(Image(str(preview_path), width=24.0 * cm, height=16.5 * cm))
    except Exception:
        try:
            from IPython.display import Image as IPyImage, display as ipy_display
            ipy_display(IPyImage(filename=str(preview_path)))
        except Exception:
            pass
    return final_preview_pdf


if str(RUN_MODE).lower() == "preview":
    preview_report_pdf = run_preview_graph_page()
else:
    if not selected_market_infos:
        raise RuntimeError("Nessun mercato selezionato per il batch.")

    market_report_rows = []
    for market_info in selected_market_infos:
        market_report_rows.append(run_independent_market_batch(market_info))

    market_reports_df = pd.DataFrame(market_report_rows)

    print("\n" + "=" * 80)
    print("BATCH COMPLETATI")
    for _, row in market_reports_df.iterrows():
        print(f"{row['Market']} | Status: {row['Status']}")
        print(f"  Report: {row.get('Report_PDF', '')}")
    print("=" * 80)

    site_reports_dir = publish_market_reports_to_site(market_reports_df)

    display(market_reports_df.drop(columns=["Strategy_Returns", "Benchmark_Returns"], errors="ignore"))



BATCH INDIPENDENTE: Canada50
Benchmark: ^GSPTSE
Ticker universo: 50
Download yfinance: Canada50 | start=2024-01-01 | end_exclusive=2026-06-20
Date range: 2024-01-02 -> 2026-06-18
Colonne scaricate: ['^GSPTSE', 'RY.TO', 'TD.TO', 'SHOP.TO', 'ENB.TO', 'BMO.TO', 'BN.TO', 'CNQ.TO', 'BNS.TO', 'CP.TO', 'CNR.TO', 'TRI.TO', 'SU.TO', 'MFC.TO', 'CSU.TO', 'ATD.TO', 'AEM.TO', 'TRP.TO', 'WCN.TO', 'CM.TO', 'SLF.TO', 'L.TO', 'IFC.TO', 'NTR.TO', 'BCE.TO', 'GWO.TO', 'POW.TO', 'IMO.TO', 'FNV.TO', 'DOL.TO', 'T.TO', 'NA.TO', 'WPM.TO', 'FTS.TO', 'QSR.TO', 'MRU.TO', 'TECK-B.TO', 'ABX.TO', 'CCO.TO', 'SAP.TO', 'EMA.TO', 'PPL.TO', 'RCI-B.TO', 'MG.TO', 'WN.TO', 'GIB-A.TO', 'BAM.TO', 'CVE.TO', 'FM.TO', 'K.TO', 'OTEX.TO']
Ticker investibili disponibili: ['RY.TO', 'TD.TO', 'SHOP.TO', 'ENB.TO', 'BMO.TO', 'BN.TO', 'CNQ.TO', 'BNS.TO', 'CP.TO', 'CNR.TO', 'TRI.TO', 'SU.TO', 'MFC.TO', 'CSU.TO', 'ATD.TO', 'AEM.TO', 'TRP.TO', 'WCN.TO', 'CM.TO', 'SLF.TO', 'L.TO', 'IFC.TO', 'NTR.TO', 'BCE.TO', 'GWO.TO', 'POW.TO', 'IMO.TO', 

,Run,trend_len,slope_min,r2_min,corr_max,corr_lookback,k_select,CAGR,Volatility,Sharpe_1pct,Information_Ratio,MaxDD,Final_Equity,Benchmark_Final_Equity,Avg_Turnover,Avg_Exposure
0,1,52,0.0,0.3,0.5,26,5,0.155061,0.133766,1.071565,-0.610147,-0.154154,1.429909,1.670168,0.137984,0.596899


PDF summary batch creato:
/content/drive/MyDrive/TradingAlgoMosaic/output/Summary_Canada50.pdf

REPORT MERCATO COMPLETATO
Report PDF: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Canada50.pdf

BATCH INDIPENDENTE: Argentina30
Benchmark: ^MERV
Ticker universo: 30
Download yfinance: Argentina30 | start=2024-01-01 | end_exclusive=2026-06-20
Date range: 2024-01-02 -> 2026-06-18
Colonne scaricate: ['^MERV', 'YPFD.BA', 'GGAL.BA', 'PAMP.BA', 'TXAR.BA', 'TGSU2.BA', 'ALUA.BA', 'COME.BA', 'BMA.BA', 'TECO2.BA', 'CEPU.BA', 'TRAN.BA', 'EDN.BA', 'MIRG.BA', 'LOMA.BA', 'SUPV.BA', 'BBAR.BA', 'VALO.BA', 'TGNO4.BA', 'CRES.BA', 'IRSA.BA', 'HARG.BA', 'MOLI.BA', 'BYMA.BA', 'METR.BA', 'AUSO.BA', 'DGCU2.BA', 'AGRO.BA', 'MORI.BA', 'SAMI.BA', 'LEDE.BA']
Ticker investibili disponibili: ['YPFD.BA', 'GGAL.BA', 'PAMP.BA', 'TXAR.BA', 'TGSU2.BA', 'ALUA.BA', 'COME.BA', 'BMA.BA', 'TECO2.BA', 'CEPU.BA', 'TRAN.BA', 'EDN.BA', 'MIRG.BA', 'LOMA.BA', 'SUPV.BA', 'BBAR.BA', 'VALO.BA', 'TGNO4.BA', 'CRES.BA', 'IRSA.BA',

,Run,trend_len,slope_min,r2_min,corr_max,corr_lookback,k_select,CAGR,Volatility,Sharpe_1pct,Information_Ratio,MaxDD,Final_Equity,Benchmark_Final_Equity,Avg_Turnover,Avg_Exposure
0,1,52,0.0,0.3,0.5,26,5,-0.073386,0.171709,-0.415184,-1.457152,-0.268005,0.82772,3.158487,0.131783,0.327132


PDF summary batch creato:
/content/drive/MyDrive/TradingAlgoMosaic/output/Summary_Argentina30.pdf

REPORT MERCATO COMPLETATO
Report PDF: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Argentina30.pdf

BATCH INDIPENDENTE: Mexico30
Benchmark: ^MXX
Ticker universo: 35
Download yfinance: Mexico30 | start=2024-01-01 | end_exclusive=2026-06-20


...

,Run,trend_len,slope_min,r2_min,corr_max,corr_lookback,k_select,CAGR,Volatility,Sharpe_1pct,Information_Ratio,MaxDD,Final_Equity,Benchmark_Final_Equity,Avg_Turnover,Avg_Exposure
0,1,52,0.0,0.3,0.5,26,5,0.615861,0.238307,2.098768,-0.162065,-0.161389,3.288534,3.511303,0.094574,0.596899


PDF summary batch creato:
/content/drive/MyDrive/TradingAlgoMosaic/output/Summary_South_Korea30.pdf

REPORT MERCATO COMPLETATO
Report PDF: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_South_Korea30.pdf

BATCH INDIPENDENTE: South Africa30
Benchmark: ^J203.JO
Ticker universo: 26
Download yfinance: South Africa30 | start=2024-01-01 | end_exclusive=2026-06-20
Date range: 2024-01-02 -> 2026-06-19
Colonne scaricate: ['^J203.JO', 'NPN.JO', 'PRX.JO', 'CFR.JO', 'BTI.JO', 'AGL.JO', 'ANH.JO', 'FSR.JO', 'SBK.JO', 'MTN.JO', 'SOL.JO', 'GFI.JO', 'ANG.JO', 'KIO.JO', 'REM.JO', 'CPI.JO', 'BID.JO', 'DSY.JO', 'VOD.JO', 'CLS.JO', 'IMP.JO', 'SHP.JO', 'MNP.JO', 'ABG.JO', 'OMU.JO', 'SLM.JO', 'NED.JO']
Ticker rimossi per spike settimanali anomali > 300%: ['ANH.JO', 'SBK.JO', 'VOD.JO', 'SLM.JO']
Ticker investibili disponibili: ['NPN.JO', 'PRX.JO', 'CFR.JO', 'BTI.JO', 'AGL.JO', 'FSR.JO', 'MTN.JO', 'SOL.JO', 'GFI.JO', 'ANG.JO', 'KIO.JO', 'REM.JO', 'CPI.JO', 'BID.JO', 'DSY.JO', 'CLS.JO', 'IMP.JO', 'SHP.J

,Run,trend_len,slope_min,r2_min,corr_max,corr_lookback,k_select,CAGR,Volatility,Sharpe_1pct,Information_Ratio,MaxDD,Final_Equity,Benchmark_Final_Equity,Avg_Turnover,Avg_Exposure
0,1,52,0.0,0.3,0.5,26,5,0.285922,0.152188,1.666729,0.614553,-0.106643,1.866105,1.530832,0.099225,0.592248


PDF summary batch creato:
/content/drive/MyDrive/TradingAlgoMosaic/output/Summary_South_Africa30.pdf

REPORT MERCATO COMPLETATO
Report PDF: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_South_Africa30.pdf

BATCH COMPLETATI
Canada50 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Canada50.pdf
Argentina30 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Argentina30.pdf
Mexico30 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Mexico30.pdf
Brazil50 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_Brazil50.pdf
South Korea30 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_South_Korea30.pdf
South Africa30 | Status: OK
  Report: /content/drive/MyDrive/TradingAlgoMosaic/output/Report_South_Africa30.pdf

SITO AGGIORNATO
Cartella report sito: /content/drive/MyDrive/TradingAlgo.it/public/reports
Report pubblicati: 6
Indice JSON: /content/drive

,Market,Benchmark,Number of Tickers,Strategy CAGR,Strategy MaxDD,Strategy Sharpe Ratio,Bench Cagr,Bench_MaxDD,Bench Sharpe,Information Ratio,PDF_Output,Summary_Chart,Capital Invested,Last Weekly Selection,Added Tickers,Removed Tickers,Selection Drivers,Status_History,Asset_Detail_Pages,Report_PDF,Summary_PDF_Temp,Detailed_PDF_Temp,Status,Error
0,Canada50,^GSPTSE,50,0.155061,-0.154154,1.071565,0.232557,-0.097241,1.895991,-0.610147,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,1.0,CVE.TO - CENOVUS ENERGY INC.<br/>TECK-B.TO - T...,-,-,"CVE.TO: trend slope 0.02, stability R2 0.90, s...","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
1,Argentina30,^MERV,30,-0.073386,-0.268005,-0.415184,0.598123,-0.399688,1.201228,-1.457152,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,1.0,VALO.BA - BANCO DE VALORES S.A.<br/>TRAN.BA - ...,-,-,"VALO.BA: trend slope 0.02, stability R2 0.80, ...","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
2,Mexico30,^MXX,34,0.187352,-0.179399,1.053479,0.082564,-0.159197,0.594825,0.693002,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,1.0,GMEXICOB.MX - GRUPO MEXICO SAB DE CV<br/>AMXB....,-,-,"GMEXICOB.MX: trend slope 0.01, stability R2 0....","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
3,Brazil50,^BVSP,41,0.061026,-0.200343,0.386502,0.103969,-0.147200,0.753431,-0.338417,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,1.0,USIM5.SA - USIMINAS PNA N1<br/>ENEV3.SA...,VBBR3.SA - VIBRA ON NM,UGPA3.SA - ULTRAPAR ON NM,"USIM5.SA: trend slope 0.02, stability R2 0.90,...","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
4,South Korea30,^KS11,29,0.615861,-0.161389,2.098768,0.668620,-0.160043,2.167249,-0.162065,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,1.0,000660.KS - SK hynix<br/>005380.KS - HyundaiMt...,017670.KS - SKTelecom,010130.KS - KOREA ZINC,"000660.KS: trend slope 0.04, stability R2 0.95...","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
5,South Africa30,^J203.JO,22,0.285922,-0.106643,1.666729,0.189556,-0.143127,1.181412,0.614553,,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,0.8,SOL.JO - Sasol Limited<br/>ANG.JO - AngloGold ...,-,-,"SOL.JO: trend slope 0.02, stability R2 0.77, s...","{'dates': ['05-01', '05-08', '05-15', '05-22',...",[/content/drive/MyDrive/TradingAlgoMosaic/outp...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,/content/drive/MyDrive/TradingAlgoMosaic/outpu...,,OK,
